# Huấn luyện mô hình TĨNH và ĐỘNG bằng TabPFN thật trên `du_lieu_co_dinh_v2_seed44` (Colab GPU)

**Mục tiêu:** có bộ kết quả cuối, nhất quán và đúng nhãn cho đồ án:
1. Mô hình **tĩnh** và **động** đều là **TabPFN thật** (không phải HistGradientBoosting), huấn luyện trên dữ liệu đã sửa `du_lieu_co_dinh_v2_seed44`.
2. Nguồn ngữ cảnh chỉ từ tập **train** (`--pool train`): ngưỡng và kiểm tra dùng cùng một mô hình, validation không lọt vào ngữ cảnh.
3. Mô hình đối chứng (CatBoost, logistic, spline, baseline lịch sử) chạy trên CPU với cùng quy trình, rồi báo cáo cuối tự tạo bảng và hình.

**Trước khi chạy:** `Runtime → Change runtime type → T4 GPU`. Upload `du_lieu_co_dinh_v2_seed44.zip` vào `/content` hoặc gốc Google Drive.
**Thời gian (ước lượng theo tốc độ đã đo ~1,3 phút mỗi lô 4.000 dòng; chưa đo cho dữ liệu động):** tĩnh khoảng 45 phút, động khoảng 60 phút (3 túi), đối chiếu CPU khoảng 15 phút.
Có thể dừng giữa chừng (Interrupt) rồi chạy lại ô huấn luyện: các lô đã xong được nạp lại (`RESUME`).
**Khuyến nghị:** chạy `MODE = 'quick'` trước (vài phút) để kiểm tra luồng, rồi đổi sang `'full'`.

**Thư mục kết quả:** `ket_qua_v2_seed44` (mỗi bộ dữ liệu một thư mục riêng để log không bị trộn), tải về `ket_qua_v2_seed44.zip`.
**Quy tắc ngưỡng chốt trước:** biên an toàn mức Thấp 0,01 và mức Cao 0,03 (`--safety-margin 0.01 --safety-margin-high 0.03`); không chỉnh gì sau khi xem test.

## 0. Cấu hình (chỉ sửa ô này)

In [ ]:
MODE = 'full'            # 'quick': dữ liệu thu nhỏ 1/8, ngữ cảnh 3.000 dòng (kiểm tra luồng, vài phút) | 'full': bản chính thức
N_BAGS = 3               # số túi ngữ cảnh TabPFN (3 túi đã tương đương 4 túi ở các lần chạy trước)
RUN_STATIC = True        # huấn luyện mô hình TĨNH bằng TabPFN
RUN_DYNAMIC = True       # huấn luyện mô hình ĐỘNG bằng TabPFN
RUN_BASELINES = True     # mô hình đối chứng (CPU) cho cả hai
RUN_REPORT = True        # báo cáo cuối (bảng, hình, BAO_CAO.md)
RUN_UNSEEN_EVAL = False  # (tuỳ chọn, rất chậm) đánh giá giảng viên/Khoa chưa thấy cho mô hình tĩnh
RESUME = True            # tiếp tục từ điểm lưu nếu đã dừng giữa chừng (cùng cấu hình, cùng thư mục)
SAVE_TO_DRIVE = True     # lưu kết quả vào Google Drive (khuyến nghị)
FIT_MODE = ''            # '' = mặc định; 'fit_with_cache' có thể nhanh hơn nhiều nhưng tốn bộ nhớ GPU: chỉ thử SAU khi quick chạy ổn

import os
QUICK = (MODE == 'quick')
if QUICK:
    BAGS, CTX = 2, '--max-context 3000'
else:
    BAGS, CTX = N_BAGS, ''
COMMON = f'--n-bags {BAGS} {CTX} ' + (f'--fit-mode {FIT_MODE}' if FIT_MODE else '')
TRAIN_EXTRA = '--no-tune --importance-rows 0 --pool train --safety-margin 0.01 --safety-margin-high 0.03' + (' --resume' if RESUME else '')
def sh(cmd):
    get_ipython().system(cmd)
print('MODE =', MODE, '| COMMON =', COMMON.strip(), '| TRAIN_EXTRA =', TRAIN_EXTRA)

## 1. Cài đặt và kiểm tra GPU

In [ ]:
!pip -q install tabpfn catboost scikit-learn pandas matplotlib tabulate
import torch, sklearn, sys
try:
    import tabpfn; tpv = getattr(tabpfn, '__version__', '?')
except Exception as e:
    tpv = None; print('KHÔNG nạp được tabpfn:', e)
gpu = torch.cuda.is_available()
print('Python', sys.version.split()[0], '| tabpfn', tpv, '| torch', torch.__version__, '| sklearn', sklearn.__version__, '| GPU:', gpu, torch.cuda.get_device_name(0) if gpu else '')
if not QUICK and (RUN_STATIC or RUN_DYNAMIC):
    assert tpv is not None, 'Cần cài được tabpfn (pip install tabpfn)'
    assert gpu, 'Bản FULL cần GPU: Runtime > Change runtime type > T4 GPU (nếu hết hạn mức GPU: chờ cấp lại hoặc đặt MODE=quick / tắt RUN_STATIC, RUN_DYNAMIC để chỉ chạy phần CPU)'

## 2. Nơi lưu kết quả

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/ket_qua_v2_seed44' + ('_quick' if QUICK else '')
else:
    OUT = '/content/ket_qua_v2_seed44' + ('_quick' if QUICK else '')
os.makedirs(OUT, exist_ok=True)
print('Kết quả lưu tại:', OUT)

In [ ]:
import shutil
if SAVE_TO_DRIVE:
    for folder in ['/content/drive/MyDrive/Colab Notebooks', '/content/drive/MyDrive']:
        z = folder + '/du_lieu_co_dinh_v2_seed44.zip'
        if os.path.exists(z) and not os.path.exists('/content/du_lieu_co_dinh_v2_seed44.zip'):
            shutil.copy(z, '/content/du_lieu_co_dinh_v2_seed44.zip'); print('Đã copy dữ liệu cố định từ', folder)
print('du_lieu_co_dinh_v2_seed44.zip có mặt:', os.path.exists('/content/du_lieu_co_dinh_v2_seed44.zip'))

## 3. Ghi các script

In [ ]:
%%writefile train_model.py
#!/usr/bin/env python3
"""
HUẤN LUYỆN MÔ HÌNH CHÍNH: TabPFN (tabular foundation model, học trong ngữ cảnh — in-context learning).
Hỗ trợ đặc trưng ma trận tác vụ–nhân sự (lecturer_task_*, dept_task_*) nếu có trong dữ liệu
(chạy build_task_personnel_matrix.py trước).

  python train_model.py train   --data ./data/ml_training_set_with_matrix.csv --out ./model [--prune]
  python train_model.py predict --model ./model/risk_model.joblib --input viec_moi.csv --output ket_qua.csv [--explain]

Khác với CatBoost: TabPFN là mạng Transformer ĐÃ HUẤN LUYỆN TRƯỚC trên hàng triệu bảng dữ liệu tổng hợp.
"Huấn luyện" ở đây KHÔNG cập nhật trọng số: các dòng huấn luyện được đưa vào làm NGỮ CẢNH (context) và mô hình
dự đoán các dòng mới bằng một lượt suy luận. Vì vậy:
  * model/risk_model.joblib lưu NGỮ CẢNH (tập dòng huấn luyện đã chọn) + bộ mã hóa; trọng số TabPFN được tải lại từ cache khi dự đoán.
  * chi phí nằm ở BƯỚC DỰ ĐOÁN (tỉ lệ với |ngữ cảnh| x |dòng cần dự đoán|) -> nên dùng GPU (Colab T4 là đủ).
  * ngữ cảnh bị giới hạn (--max-context; mặc định ~10.000 dòng, v2.5+ trên GPU ~50.000 dòng). Có thể dùng nhiều "túi" ngữ cảnh khác nhau (--n-bags).

Quy trình: chia THEO THỜI GIAN 60/15/25 -> chọn siêu tham số trên validation -> dựng ngữ cảnh từ train+validation
-> đánh giá trên tập test (khoảng tin cậy bootstrap) -> chọn ngưỡng cảnh báo theo mục tiêu nghiệp vụ trên validation.

Cần: pip install tabpfn   (lần đầu tải trọng số từ Hugging Face; v2.5 trở lên cần chấp nhận giấy phép + HF token)
"""
import argparse, json, os, warnings
os.environ.setdefault("OMP_NUM_THREADS", "1")  # Tránh overhead khởi tạo OpenMP (~1.4s) trên Windows khi chạy trên CPU
import numpy as np, pandas as pd, joblib
import sklearn
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
warnings.filterwarnings("ignore")

try:
    import torch
except ImportError:
    torch = None

try:
    from tabpfn import TabPFNClassifier
    try:
        from tabpfn.constants import ModelVersion
    except Exception:  # bản tabpfn cũ không có ModelVersion
        ModelVersion = None
except ImportError:
    TabPFNClassifier = None
    ModelVersion = None

ID_COLS = ["task_id", "lecturer_id", "faculty", "assigned_date"]
TARGETS = ["is_late", "late_vs_original_deadline", "late_vs_current_deadline"]
LABEL = "is_late"
LEVEL_NAMES = ["Thấp", "Trung bình", "Cao"]
DEFAULT_THR = (0.20, 0.50)
# Cấu hình TabPFN & chế độ CPU nhanh (đặt bằng tham số dòng lệnh; xem main)
TABPFN_VERSION = "v2"     # v2 (mở, không cần đăng nhập) | v2.5 | v2.6 | ... (cần chấp nhận giấy phép trên Hugging Face)
DEVICE = "auto"           # auto | cpu | cuda
MAX_CONTEXT = None        # số dòng ngữ cảnh tối đa mỗi túi; None -> 10.000 (v2 hoặc CPU) / 50.000 (v2.5+ trên GPU)
N_BAGS = 1                # số túi ngữ cảnh độc lập (trung bình xác suất)
CONTEXT_MODE = "random"   # random: lấy ngẫu nhiên | recent: ưu tiên các dòng mới nhất theo thời gian
PRED_BATCH = 4000         # số dòng dự đoán mỗi lượt (giới hạn bộ nhớ)
FIT_MODE = None           # None | low_memory | fit_preprocessors | fit_with_cache (nhanh hơn khi dự đoán nhiều lô cùng ngữ cảnh, tốn bộ nhớ GPU)
CPU_LIGHT_CONTEXT_CAP = 2048   # --cpu-light: chạy TabPFN (không phải mô hình khác) trên CPU với 1 túi x 2048 dòng ngữ cảnh gần nhất; CHƯA đánh giá chất lượng
# Lưới siêu tham số gọn (chọn theo log-loss validation). TabPFN gần như không cần chỉnh tay.
PARAM_GRID = [
    dict(n_estimators=8, softmax_temperature=0.9),
    dict(n_estimators=8, softmax_temperature=1.0),
    dict(n_estimators=16, softmax_temperature=0.9),
]
GROUPS = {
    "giảng viên": ["academic_rank", "degree", "department", "management_role", "years_experience", "tenure_days"],
    "tải giảng dạy": ["teaching_hours_semester", "course_count", "student_count", "supervised_student_count"],
    "lịch năm học": ["academic_period", "semester", "academic_week", "peak_season", "accreditation_flag", "holiday_in_window"],
    "mô tả công việc": ["task_group", "task_type", "task_subtype", "assignment_level", "assignment_mode", "assignment_channel",
                        "team_size", "require_approval", "converted_workload", "converted_workload_missing"],
    "thời hạn": ["available_days", "working_days_available", "expected_duration_days", "deadline_pressure",
                 "is_urgent_assignment", "estimated_effort_hours", "deadline_month_end"],
    "lịch sử giảng viên": ["hist_count", "hist_late_rate", "hist_avg_delay", "hist_early_rate", "hist_avg_delay_missing",
                           "hist_early_rate_missing", "is_cold_start", "hist_late_last5", "hist_late_last5_missing", "late_last_30d"],
    "tải hiện tại": ["active_tasks", "active_high_priority_tasks", "overdue_open_tasks", "deadlines_next_7d",
                     "deadlines_next_14d", "workload_next_14d"],
    "độ phức tạp & ưu tiên": ["complexity", "complexity_num", "priority", "priority_num", "subtask_count",
                              "dependency_count", "require_external_input"],
    "ma trận tác vụ–nhân sự": ["lecturer_task_count", "lecturer_task_late_rate", "dept_task_late_rate",
                               "lecturer_task_late_lift", "task_matrix_source"],
    "tiến độ": ["progress_pct", "progress_gap", "expected_progress_pct", "days_elapsed", "days_remaining",
                "checkpoint_fraction", "progress_missing"],
}
NAMES = {
    "deadline_pressure": "áp lực deadline", "active_tasks": "số việc đang mở",
    "workload_next_14d": "khối lượng đến hạn trong 14 ngày (giờ)",
    "deadlines_next_7d": "số hạn trong 7 ngày tới", "deadlines_next_14d": "số hạn trong 14 ngày tới",
    "hist_late_rate": "tỉ lệ trễ lịch sử", "complexity": "độ phức tạp", "priority": "mức ưu tiên",
    "available_days": "số ngày được giao", "expected_duration_days": "thời lượng điển hình (ngày)",
    "dependency_count": "số bên phụ thuộc", "subtask_count": "số đầu việc con",
    "require_external_input": "cần đầu vào bên ngoài", "overdue_open_tasks": "số việc đang mở đã quá hạn",
    "active_high_priority_tasks": "số việc ưu tiên cao đang mở", "years_experience": "kinh nghiệm (năm)",
    "tenure_days": "số ngày công tác tại Khoa", "teaching_hours_semester": "giờ giảng học kỳ",
    "academic_period": "giai đoạn năm học", "progress_gap": "chênh lệch tiến độ so với kỳ vọng (%)",
    "task_type": "loại công việc", "management_role": "vai trò quản lý",
    "hist_count": "số việc đã hoàn thành trước đây", "hist_late_last5": "tỉ lệ trễ 5 việc gần nhất",
    "late_last_30d": "số việc trễ trong 30 ngày qua", "deadline_month_end": "hạn chót cuối tháng",
    "progress_missing": "chưa cập nhật tiến độ", "student_count": "số sinh viên phụ trách",
    "complexity_num": "độ phức tạp (0-2)", "priority_num": "mức ưu tiên (0-3)",
    "course_count": "số học phần", "supervised_student_count": "số SV hướng dẫn",
    "hist_avg_delay": "số ngày trễ TB trước đây", "hist_early_rate": "tỉ lệ hoàn thành sớm trước đây",
    "lecturer_task_late_rate": "tỉ lệ trễ GV với loại việc này",
    "lecturer_task_count": "số lần GV đã làm loại việc này",
    "dept_task_late_rate": "tỉ lệ trễ Bộ môn với loại việc",
    "lecturer_task_late_lift": "lệch tỉ lệ trễ GV so với Bộ môn",
    "task_matrix_source": "nguồn prior ma trận tác vụ–nhân sự",
}


def engineer(df):
    df = df.copy()
    df = df.drop(columns=[c for c in df.columns if c.startswith("log_")], errors="ignore")
    if "complexity" in df:
        df["complexity_num"] = df.complexity.map({"Thấp": 0, "Trung bình": 1, "Cao": 2})
    if "priority" in df:
        df["priority_num"] = df.priority.map({"Thấp": 0, "Trung bình": 1, "Cao": 2, "Khẩn cấp": 3})
    # Tự động tính bổ sung các cột tiến độ động (dynamic progress) nếu người dùng chỉ truyền progress_pct và days_elapsed
    if "progress_pct" in df.columns or "checkpoint_fraction" in df.columns:
        if "days_elapsed" in df.columns and "available_days" in df.columns:
            avail = df["available_days"].clip(lower=1)
            if "days_remaining" not in df.columns:
                df["days_remaining"] = (avail - df["days_elapsed"]).clip(lower=0)
            if "checkpoint_fraction" not in df.columns:
                df["checkpoint_fraction"] = (df["days_elapsed"] / avail).round(2)
            if "expected_progress_pct" not in df.columns:
                df["expected_progress_pct"] = (100.0 * df["days_elapsed"] / avail).clip(0, 100).round(1)
        elif "checkpoint_fraction" in df.columns and "available_days" in df.columns:
            avail = df["available_days"].clip(lower=1)
            if "days_elapsed" not in df.columns:
                df["days_elapsed"] = np.maximum(1, np.round(df["checkpoint_fraction"] * avail)).astype(int)
            if "days_remaining" not in df.columns:
                df["days_remaining"] = (avail - df["days_elapsed"]).clip(lower=0)
            if "expected_progress_pct" not in df.columns:
                df["expected_progress_pct"] = (100.0 * df["days_elapsed"] / avail).clip(0, 100).round(1)
        if "progress_pct" in df.columns:
            if "progress_missing" not in df.columns:
                df["progress_missing"] = df["progress_pct"].isna().astype(int)
            if "expected_progress_pct" in df.columns and "progress_gap" not in df.columns:
                df["progress_gap"] = (df["progress_pct"] - df["expected_progress_pct"]).round(1)
    # Ba cờ "thiếu lịch sử" trùng hệt is_cold_start (tương quan = 1,0) -> bỏ bớt để tránh đặc trưng lặp
    if "is_cold_start" in df:
        for c in DUP_FLAGS:
            if c in df and df[c].equals(df["is_cold_start"]):
                df = df.drop(columns=[c])
    return df


DUP_FLAGS = ["hist_avg_delay_missing", "hist_early_rate_missing", "hist_late_last5_missing"]


def restore_dups(eng, features):
    """Mô hình cũ có thể còn các cờ trùng lặp trong danh sách đặc trưng: khôi phục từ is_cold_start để dự đoán không bị lệch."""
    for c in features:
        if c in DUP_FLAGS and c not in eng.columns and "is_cold_start" in eng.columns:
            eng[c] = eng["is_cold_start"]
    return eng


def split_cols(X):
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    return cat, [c for c in X.columns if c not in cat]


def resolve_device():
    if DEVICE != "auto":
        return DEVICE
    return "cuda" if (torch is not None and torch.cuda.is_available()) else "cpu"


def context_limit():
    if MAX_CONTEXT:
        return int(MAX_CONTEXT)
    # v2 được huấn luyện trước tới ~10.000 dòng; v2.5 trở lên tới ~50.000 dòng (cần GPU)
    big = TABPFN_VERSION != "v2" and resolve_device() == "cuda"
    return 50000 if big else 10000


def fit_encoder(X):
    """Mã hóa thứ tự cho cột phân loại (TabPFN nhận số + chỉ số cột phân loại); NaN giữ nguyên."""
    cat, num = split_cols(X)
    maps = {}
    for c in cat:
        vals = X[c].dropna().astype(str).unique()
        maps[c] = {v: i for i, v in enumerate(sorted(vals))}
    return dict(cat=cat, num=num, maps=maps, order=list(X.columns))


def apply_encoder(enc, X):
    if getattr(X, "_is_encoded", False):
        return X
    out = pd.DataFrame(index=X.index)
    for c in enc["order"]:
        if c not in X.columns:
            out[c] = np.nan
        elif c in enc["maps"]:
            out[c] = X[c].map(lambda v, m=enc["maps"][c]: m.get(str(v), np.nan) if v == v and v is not None else np.nan)
        else:
            out[c] = pd.to_numeric(X[c], errors="coerce")
    res = out.astype(np.float32)
    res._is_encoded = True
    return res


def make_tabpfn(params, cat_idx, seed=0):  # noqa
    if TabPFNClassifier is None:
        raise SystemExit("Chưa cài tabpfn. Chạy: pip install tabpfn")
    kw = dict(
        n_estimators=params.get("n_estimators", 8),
        softmax_temperature=params.get("softmax_temperature", 0.9),
        categorical_features_indices=cat_idx or None,
        device=resolve_device(),
        ignore_pretraining_limits=True,
        random_state=seed,
    )
    if FIT_MODE:
        kw["fit_mode"] = FIT_MODE
    if ModelVersion is not None and hasattr(TabPFNClassifier, "create_default_for_version"):
        return TabPFNClassifier.create_default_for_version(ModelVersion(TABPFN_VERSION), **kw)
    return TabPFNClassifier(**kw)


def pick_context(n, y, k, seed):
    """Chọn chỉ số dòng làm ngữ cảnh cho túi thứ k."""
    lim = context_limit()
    if n <= lim:
        return np.arange(n)
    if CONTEXT_MODE == "recent":
        # dữ liệu đã sắp theo thời gian: nửa ngữ cảnh là các dòng mới nhất, nửa còn lại lấy ngẫu nhiên từ phần cũ hơn
        half = lim // 2
        rng = np.random.default_rng(seed + 1000 * k)
        older = rng.choice(n - half, size=lim - half, replace=False)
        return np.sort(np.concatenate([older, np.arange(n - half, n)]))
    rng = np.random.default_rng(seed + 1000 * k)
    return np.sort(rng.choice(n, size=lim, replace=False))


def get_clf(bundle, k, max_ctx=None):
    """Dựng (lười) bộ phân loại TabPFN cho túi k từ ngữ cảnh đã lưu; cache trong bundle['_clf']."""
    cache = bundle.setdefault("_clf", {})
    ckey = (k, max_ctx)
    if ckey not in cache:
        Xc, yc = bundle["contexts"][k]
        if max_ctx and len(Xc) > max_ctx:
            # lấy các dòng cuối (gần nhất theo thời gian vì ngữ cảnh đã sort)
            Xc, yc = Xc[-max_ctx:], yc[-max_ctx:]
        clf = make_tabpfn(bundle["hyperparams"], bundle["cat_idx"], seed=bundle.get("seed", 0) + k)
        clf.fit(Xc, yc)
        cache[ckey] = clf
    return cache[ckey]


def prob_detail(model, X, ckpt=None, tag=None):
    """Xác suất trễ hạn: (trung bình các túi, độ lệch chuẩn giữa các túi). Độ lệch = 0 nếu chỉ có 1 túi.
    Nếu truyền ckpt (thư mục) và tag: mỗi lô (túi, đoạn dòng) được lưu ra đĩa ngay khi tính xong và được nạp lại nếu đã có
    -> dừng giữa chừng rồi chạy lại chỉ mất tối đa một lô."""
    import time
    Xe = apply_encoder(model["encoder"], X)
    n = len(Xe)
    if n == 0:
        return np.zeros(0), np.zeros(0)

    n_bags = 1 if (MAX_CONTEXT and MAX_CONTEXT <= CPU_LIGHT_CONTEXT_CAP and resolve_device() == "cpu" and N_BAGS == 1) else len(model["contexts"])
    ctx_cap = int(MAX_CONTEXT) if MAX_CONTEXT else None
    chunks_per_bag = max(1, -(-n // PRED_BATCH))
    total = n_bags * chunks_per_bag
    per_bag = []
    n_cached = n_done = 0
    t0 = time.time()
    for k in range(n_bags):
        clf = None
        ps = []
        for ci, i in enumerate(range(0, n, PRED_BATCH)):
            path = os.path.join(ckpt, f"{tag}_b{k}_c{ci}.npy") if ckpt and tag else None
            if path and os.path.exists(path):
                ps.append(np.load(path))
                n_cached += 1
                continue
            if clf is None:
                clf = get_clf(model, k, max_ctx=ctx_cap)
            part = clf.predict_proba(Xe.iloc[i:i + PRED_BATCH].values)[:, 1]
            if path:
                tmp = path + ".tmp.npy"
                np.save(tmp, part)
                os.replace(tmp, path)
            ps.append(part)
            n_done += 1
            if tag and total > 1:
                el = time.time() - t0
                left = (total - n_done - n_cached) * el / n_done
                print(f"   [{tag}] lô {n_done + n_cached}/{total} (túi {k + 1}/{n_bags}) | đã chạy {el / 60:.1f} phút | còn khoảng {left / 60:.1f} phút", flush=True)
        per_bag.append(np.concatenate(ps) if ps else np.zeros(0))
    if ckpt and tag:
        print(f"   [{tag}] {n_done} lô vừa tính, {n_cached} lô nạp lại từ điểm lưu", flush=True)
    arr = np.vstack(per_bag)
    mean = np.clip(arr.mean(axis=0), 1e-4, 1 - 1e-4)
    sd = arr.std(axis=0) if arr.shape[0] > 1 else np.zeros(n)
    return mean, sd


def prob(model, X):
    """Xác suất trễ hạn: trung bình các túi ngữ cảnh."""
    return prob_detail(model, X)[0]


def ece(y, p, k=10):
    d = pd.DataFrame({"y": y, "p": p, "q": pd.qcut(p, k, duplicates="drop")})
    g = d.groupby("q", observed=True).agg(y=("y", "mean"), p=("p", "mean"), n=("y", "size"))
    return float((g.n * (g.y - g.p).abs()).sum() / g.n.sum())


def metrics(y, p):
    top = p >= np.quantile(p, 0.8)
    return dict(
        roc_auc=roc_auc_score(y, p),
        pr_auc=average_precision_score(y, p),
        brier=brier_score_loss(y, p),
        ece=ece(y, p),
        base_rate=float(y.mean()),
        recall_at_top20pct=float(y[top].sum() / max(1, y.sum())),
        precision_at_top20pct=float(y[top].mean()),
    )


def boot_auc(y, p, groups=None, n=300, seed=0):
    """Khoảng tin cậy 95% bootstrap của AUC.
    Nếu truyền `groups` (ví dụ lecturer_id) thì lấy mẫu lại THEO CỤM: các dòng của cùng một giảng viên không độc lập,
    nên lấy mẫu theo từng dòng làm khoảng tin cậy hẹp hơn thực tế."""
    y = np.asarray(y)
    p = np.asarray(p)
    rng = np.random.default_rng(seed)
    a = []
    if groups is None:
        for _ in range(n):
            i = rng.integers(0, len(y), len(y))
            if y[i].min() != y[i].max():
                a.append(roc_auc_score(y[i], p[i]))
    else:
        g = np.asarray(groups)
        order = np.argsort(g, kind="stable")
        ug, start = np.unique(g[order], return_index=True)
        bounds = list(start) + [len(g)]
        members = [order[bounds[k]:bounds[k + 1]] for k in range(len(ug))]
        for _ in range(n):
            pick = rng.integers(0, len(ug), len(ug))
            i = np.concatenate([members[k] for k in pick])
            if y[i].min() != y[i].max():
                a.append(roc_auc_score(y[i], p[i]))
    return [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))]


def wilson_bounds(k, n, z=1.96):
    """Cận dưới/trên (Wilson, ~95%) của một tỉ lệ k/n."""
    if n == 0:
        return 0.0, 1.0
    ph = k / n
    den = 1 + z * z / n
    c = (ph + z * z / (2 * n)) / den
    h = z * np.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / den
    return float(c - h), float(c + h)


def pick_thresholds(y, p, max_late_low=0.10, min_prec_high=0.60, min_n=30, safety_margin=0.01, min_band=0.05,
                    cao_top_pct=None, safety_margin_high=None):
    """Chọn hai ngưỡng (Thấp | Trung bình | Cao) trên validation, có BIÊN AN TOÀN:
      - Thấp: tỉ lệ trễ của nhóm p < t1 phải <= max_late_low, tính theo CẬN TRÊN 95% (Wilson) cộng thêm safety_margin
              (ngưỡng chọn vừa khít trên một lát validation thường kém đi khi sang giai đoạn sau).
      - Cao: độ chính xác của nhóm p >= t2 phải >= min_prec_high, tính theo CẬN DƯỚI 95% trừ safety_margin_high
              (mặc định bằng safety_margin; thử lùi theo thời gian cho thấy mức Cao cần biên lớn hơn mức Thấp).
      - Dải 'Trung bình' tối thiểu min_band (nới t1 xuống / t2 lên; việc này chỉ làm hai nhóm biên tinh hơn, không làm mất đảm bảo).
      - cao_top_pct (tuỳ chọn, ví dụ 10): thay t2 bằng ngưỡng để đúng top x% việc rơi vào 'Cao' (theo năng lực xử lý).
    Trả về (ngưỡng, đạt_mục_tiêu)."""
    y = np.asarray(y)
    p = np.asarray(p)
    sm_hi = safety_margin if safety_margin_high is None else safety_margin_high
    grid = np.linspace(0.02, 0.95, 94)
    t1 = t2 = None
    for t in grid:
        s_ = p < t
        n = int(s_.sum())
        if n < min_n:
            continue
        if wilson_bounds(int(y[s_].sum()), n)[1] + safety_margin <= max_late_low:
            t1 = float(t)
        else:
            break      # dừng ở lần vi phạm đầu tiên: không nhảy sang ngưỡng lớn hơn chỉ vì nhiễu làm đạt lại
    if cao_top_pct:
        t2 = float(np.quantile(p, 1 - cao_top_pct / 100.0))
    else:
        for t in grid:
            s_ = p >= t
            n = int(s_.sum())
            if n >= min_n and wilson_bounds(int(y[s_].sum()), n)[0] - sm_hi >= min_prec_high:
                t2 = float(t)
                break
    if t1 is None or t2 is None:
        return list(DEFAULT_THR), False
    if t2 - t1 < min_band:
        gap = (min_band - (t2 - t1)) / 2
        t1, t2 = t1 - gap, t2 + gap
    if t1 <= 0 or t1 >= t2:
        return list(DEFAULT_THR), False
    return [round(t1, 2), round(t2, 2)], True


def levels_from(thr):
    return [(0.0, thr[0], LEVEL_NAMES[0]), (thr[0], thr[1], LEVEL_NAMES[1]), (thr[1], 1.01, LEVEL_NAMES[2])]


def level_of(p, levels):
    return next(n for lo, hi, n in levels if lo <= p < hi)


def baseline_values(X):
    cat, num = split_cols(X)
    b = {}
    for c in num:
        b[c] = float(X[c].median()) if X[c].notna().any() else 0.0
    for c in cat:
        b[c] = X[c].mode().iloc[0] if X[c].notna().any() else ""
    return b


NAMES.update({
    "task_group": "nhóm công việc", "task_subtype": "loại và quy mô công việc", "assignment_level": "cấp giao việc",
    "assignment_mode": "hình thức giao (cá nhân/nhóm)", "assignment_channel": "kênh giao việc", "team_size": "số người thực hiện",
    "is_urgent_assignment": "giao gấp (áp lực deadline > 1,25)", "require_approval": "cần phê duyệt",
    "holiday_in_window": "có nghỉ lễ/Tết trong thời gian thực hiện", "converted_workload": "giờ chuẩn quy đổi",
    "semester": "học kỳ", "academic_week": "tuần của năm học", "peak_season": "mùa cao điểm (thi/chấm/nộp điểm)",
    "accreditation_flag": "trong đợt kiểm định", "academic_rank": "chức danh", "degree": "học vị", "department": "bộ môn",
    "converted_workload_missing": "thiếu giờ chuẩn quy đổi", "hist_avg_delay_missing": "thiếu số ngày trễ TB trước đây",
    "hist_early_rate_missing": "thiếu tỉ lệ làm sớm trước đây", "hist_late_last5_missing": "thiếu tỉ lệ trễ 5 việc gần nhất",
    "is_cold_start": "giảng viên mới (ít lịch sử)",
    "progress_pct": "tiến độ đã đạt (%)", "expected_progress_pct": "tiến độ kỳ vọng (%)",
    "days_elapsed": "số ngày đã thực hiện", "days_remaining": "số ngày còn lại tới hạn",
    "checkpoint_fraction": "mốc kiểm tra tiến độ",
})


def fmt(v):
    """Hiển thị gọn: số nguyên giữ nguyên, số lớn làm tròn, số nhỏ 3 chữ số có nghĩa."""
    if v is None or (isinstance(v, float) and v != v) or (isinstance(v, np.floating) and v != v):
        return "thiếu"
    if isinstance(v, (int, float, np.number)) and v == v:
        v = float(v)
        if v == int(v):
            return str(int(v))
        return f"{v:.0f}" if abs(v) >= 100 else f"{v:.3g}"
    return str(v)


def fmt_feat(X, i, f):
    """Hiển thị tên và giá trị đặc trưng; nếu progress_gap bị thiếu do chưa cập nhật tiến độ thì ghi rõ."""
    val = X.at[i, f]
    if f == "progress_gap" and (val is None or (isinstance(val, (float, np.floating)) and val != val)):
        if "progress_missing" in X.columns and X.at[i, "progress_missing"] == 1:
            return "chưa cập nhật tiến độ = 1"
    return f"{NAMES.get(f, f)} = {fmt(val)}"


# Các đặc trưng thực chất là một yếu tố (bản chữ + bản số, hoặc giá trị + cờ báo thiếu / cold-start) -> gộp khi giải thích
EXPLAIN_MERGE = {
    "complexity": ["complexity_num"],
    "priority": ["priority_num"],
    "converted_workload": ["converted_workload_missing"],
    "hist_avg_delay": ["hist_avg_delay_missing"],
    "hist_early_rate": ["hist_early_rate_missing"],
    "hist_late_last5": ["hist_late_last5_missing"],
    "hist_count": ["is_cold_start"],
    "lecturer_task_count": ["task_matrix_source"],
    "progress_gap": ["progress_pct", "progress_missing"],
    "checkpoint_fraction": ["expected_progress_pct", "days_elapsed", "days_remaining"],
}


def explain(m, X, k_up=3, k_down=2, min_effect=0.02):
    """Đóng góp xấp xỉ: Δ logit khi đặt từng yếu tố về giá trị điển hình (trung vị/mốt của dữ liệu huấn luyện).
    Mọi biến thể (n dòng x số yếu tố) được GỘP thành MỘT lần suy luận để tiết kiệm thời gian.
    Lưu ý: 'tăng/giảm rủi ro' là SO VỚI MỨC ĐIỂN HÌNH, không phải mức rủi ro tuyệt đối; các đóng góp không cộng lại đúng bằng
    tổng thay đổi vì các đặc trưng tương quan với nhau; đây là gợi ý yếu tố nổi bật, không chứng minh quan hệ nhân quả."""
    X = X.reset_index(drop=True)
    features, baseline = m["features"], m["baseline"]
    merged = {x for v in EXPLAIN_MERGE.values() for x in v}
    units = []  # (tên hiển thị, [các cột cùng thay])
    for f in features:
        if f in merged or f not in X.columns or f not in baseline:
            continue
        cols = [f] + [c for c in EXPLAIN_MERGE.get(f, []) if c in X.columns and c in baseline]
        units.append((f, cols))
    n = len(X)

    def logit(p):
        p = np.clip(p, 1e-4, 1 - 1e-4)
        return np.log(p / (1 - p))

    # Mã hóa X và baseline một lần sang dạng số trước khi nhân bản 50 khối -> tăng tốc suy luận + giải thích gấp ~30 lần (< 35ms)
    enc = m["encoder"]
    Xe0 = apply_encoder(enc, X)
    base_df = pd.DataFrame([{k: baseline.get(k, np.nan) for k in enc["order"]}])
    base_enc = apply_encoder(enc, base_df).iloc[0]

    blocks = [Xe0]
    for f, cols in units:
        Xb = Xe0.copy()
        for c in cols:
            if c in Xb.columns:
                Xb[c] = base_enc[c]
        # giữ tính nhất quán của hiệu số ma trận khi thay lecturer_task_late_rate hoặc dept_task_late_rate về baseline
        if f in ("lecturer_task_late_rate", "dept_task_late_rate") and {"lecturer_task_late_rate", "dept_task_late_rate", "lecturer_task_late_lift"} <= set(Xb.columns):
            Xb["lecturer_task_late_lift"] = Xb["lecturer_task_late_rate"] - Xb["dept_task_late_rate"]
        blocks.append(Xb)
    big = pd.concat(blocks, ignore_index=True)
    big._is_encoded = True
    lg = logit(prob(m, big))
    l0 = lg[:n]
    C = pd.DataFrame({f: l0 - lg[(j + 1) * n:(j + 2) * n] for j, (f, _) in enumerate(units)})
    out = []
    for i in range(n):
        row = C.iloc[i].sort_values(ascending=False)
        up = row[row > min_effect].head(k_up)
        down = row[row < -min_effect].sort_values().head(k_down)
        txt = lambda s: "; ".join(fmt_feat(X, i, f) for f in s.index)
        parts = []
        if len(up):
            parts.append("Tăng rủi ro (so với mức điển hình): " + txt(up))
        if len(down):
            parts.append("Giảm rủi ro: " + txt(down))
        out.append(" | ".join(parts) if parts else "Không có yếu tố nổi bật so với mức điển hình")
    return out


def fit_model(X, y, params=None, seed=0):
    """"Huấn luyện" TabPFN = chọn ngữ cảnh + mã hóa. X phải được sắp theo thời gian (cũ -> mới).
    Trả về bundle dùng được với prob()."""
    X = X.reset_index(drop=True)
    y = np.asarray(y).astype(int)
    params = params or PARAM_GRID[0]
    enc = fit_encoder(X)
    Xe = apply_encoder(enc, X)
    cat_idx = [Xe.columns.get_loc(c) for c in enc["cat"] if c in Xe.columns]
    contexts = []
    for k in range(max(1, N_BAGS)):
        idx = pick_context(len(Xe), y, k, seed)
        contexts.append((Xe.iloc[idx].values, y[idx]))
    return dict(
        model_name="tabpfn", tabpfn_version=TABPFN_VERSION, features=list(X.columns), hyperparams=params,
        encoder=enc, contexts=contexts, cat_idx=cat_idx, seed=seed,
        cat_features=[c for c in enc["cat"]], context_mode=CONTEXT_MODE, max_context=context_limit(),
    )


def permutation_importance(bundle, X, y, n_rows=1500, seed=0):
    """Độ giảm AUC khi xáo trộn từng đặc trưng (trên mẫu con của validation) — TabPFN không có importance nội tại."""
    rng = np.random.default_rng(seed)
    i = rng.choice(len(X), size=min(n_rows, len(X)), replace=False)
    Xs, ys = X.iloc[i].reset_index(drop=True), np.asarray(y)[i]
    base = roc_auc_score(ys, prob(bundle, Xs))
    rows = []
    for f in bundle["features"]:
        if f not in Xs.columns:
            continue
        Xp = Xs.copy()
        Xp[f] = Xp[f].values[rng.permutation(len(Xp))]
        rows.append((f, base - roc_auc_score(ys, prob(bundle, Xp))))
    imp = pd.DataFrame(rows, columns=["feature", "auc_drop"]).sort_values("auc_drop", ascending=False)
    imp["importance"] = imp["auc_drop"]
    return imp[["feature", "importance", "auc_drop"]]


def unseen_categories(enc, X, max_examples=3):
    """Giá trị phân loại có mặt trong dữ liệu mới nhưng chưa từng xuất hiện lúc huấn luyện (sẽ bị coi là thiếu)."""
    out = {}
    for c, mp in enc["maps"].items():
        if c not in X.columns:
            continue
        v = X[c].dropna().astype(str)
        bad = v[~v.isin(mp.keys())]
        if len(bad):
            out[c] = dict(n=int(len(bad)), pct=round(100 * len(bad) / max(1, len(X)), 1), examples=sorted(bad.unique())[:max_examples])
    return out


def make_meta(df_all, trva_mask, cols, args_used):
    """Thông tin để biết khi nào cần huấn luyện lại và kiểm tra tương thích khi nạp mô hình."""
    import datetime, platform
    d = df_all.loc[trva_mask, "assigned_date"]
    try:
        import tabpfn as _tp
        tp_ver = getattr(_tp, "__version__", "?")
    except Exception:
        tp_ver = "?"
    return dict(
        trained_at=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        data_start=str(d.min().date()), data_end=str(d.max().date()), n_rows=int(trva_mask.sum()),
        n_features=len(cols), feature_schema=list(cols),
        versions=dict(python=platform.python_version(), tabpfn=str(tp_ver), torch=str(getattr(torch, "__version__", "?")), sklearn=sklearn.__version__),
        settings=args_used,
    )


def slim(bundle):
    """Bản để lưu: bỏ các bộ phân loại TabPFN đã dựng (sẽ dựng lại từ ngữ cảnh khi nạp)."""
    return {k: v for k, v in bundle.items() if k not in ("_clf", "_fast_clf", "fast_models")}


_nn = lambda v: None if v != v else round(float(v), 4)


class _Tee:
    """Ghi mọi thứ in ra màn hình đồng thời vào file (để bảng kết quả không mất khi đóng phiên Colab)."""
    def __init__(self, stream, path):
        self.stream, self.f = stream, open(path, "a", encoding="utf-8")

    def write(self, x):
        self.stream.write(x)
        self.f.write(x)
        self.f.flush()
        return len(x)

    def flush(self):
        self.stream.flush()
        self.f.flush()

    def close(self):
        self.f.close()


def train(a):
    import sys
    os.makedirs(a.out, exist_ok=True)
    _orig = sys.stdout
    sys.stdout = _Tee(_orig, os.path.join(a.out, "train_report.txt"))
    print("\n" + "=" * 70 + "\nLẦN CHẠY train — " + pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S") + "\n" + "=" * 70)
    try:
        return _train(a)
    finally:
        sys.stdout.close()
        sys.stdout = _orig


def _train(a):
    for k, v in dict(max_late_low=0.10, min_prec_high=0.60, safety_margin=0.01, min_band=0.05, cao_top_pct=None, safety_margin_high=None,
                     resume=False, keep_checkpoint=False, pool="train").items():
        if not hasattr(a, k):
            setattr(a, k, v)
    if TabPFNClassifier is None:
        raise SystemExit("[DỪNG] Chưa cài tabpfn. Huấn luyện mô hình chính cần TabPFN + GPU (pip install tabpfn; Colab T4).")
    use_tabpfn = True
    df = engineer(pd.read_csv(a.data, parse_dates=["assigned_date"])).sort_values("assigned_date").reset_index(drop=True)
    y = df[LABEL].astype(int).values
    X = df.drop(columns=[c for c in ID_COLS + TARGETS if c in df.columns])
    t1, t2 = df.assigned_date.quantile(0.60), df.assigned_date.quantile(0.75)
    tr = (df.assigned_date <= t1).values
    va = ((df.assigned_date > t1) & (df.assigned_date <= t2)).values
    te = (df.assigned_date > t2).values
    trva = tr | va
    print(
        f"Train {tr.sum()} (<= {t1.date()}) | Validation {va.sum()} | Test {te.sum()} (> {t2.date()}, "
        f"{int(y[te].sum())} việc trễ) | {X.shape[1]} đặc trưng | mô hình: TabPFN {TABPFN_VERSION} | "
        f"thiết bị: {resolve_device()} | ngữ cảnh tối đa {context_limit():,} dòng x {N_BAGS} túi ({CONTEXT_MODE})"
    )
    if resolve_device() == "cpu":
        print("[LƯU Ý] Đang chạy CPU: dự đoán TabPFN chậm. Nên dùng GPU (Colab: Runtime > Change runtime type > T4) "
              "hoặc giảm --max-context / dùng --no-tune.")

    # --- điểm lưu (checkpoint): cho phép dừng giữa chừng rồi chạy lại với --resume ---
    ckpt = os.path.join(a.out, "checkpoint")
    key = dict(data=os.path.basename(a.data), n_rows=int(len(df)), n_late=int(y.sum()), tabpfn=TABPFN_VERSION,
               n_bags=N_BAGS, max_context=context_limit(), mode=CONTEXT_MODE, batch=PRED_BATCH, no_tune=bool(a.no_tune),
               grid=[sorted(p.items()) for p in (PARAM_GRID[:1] if a.no_tune else PARAM_GRID)], n_features=int(X.shape[1]))
    os.makedirs(a.out, exist_ok=True)
    keyfile = os.path.join(ckpt, "key.json")
    if os.path.isdir(ckpt):
        same = False
        if a.resume and os.path.exists(keyfile):
            try:
                same = json.load(open(keyfile, encoding="utf-8")) == json.loads(json.dumps(key))
            except Exception:
                same = False
        if not same:
            if a.resume:
                print("[LƯU Ý] Điểm lưu cũ không khớp cấu hình/dữ liệu hiện tại -> bỏ và tính lại từ đầu.")
            import shutil
            shutil.rmtree(ckpt, ignore_errors=True)
        else:
            print("Tiếp tục từ điểm lưu: " + ckpt)
    os.makedirs(ckpt, exist_ok=True)
    json.dump(key, open(keyfile, "w", encoding="utf-8"), ensure_ascii=False)

    all_cols = list(X.columns)
    cols = all_cols
    pruned = []
    grid = PARAM_GRID[:1] if a.no_tune else PARAM_GRID

    def select(cols_, grid_, use_ckpt=True):
        best = None
        for gi, params in enumerate(grid_):
            m = fit_model(X.loc[tr, cols_], y[tr], params=params)
            p = prob_detail(m, X.loc[va, cols_], ckpt=ckpt if use_ckpt else None, tag=f"va{gi}")[0]
            ll = log_loss(y[va], p)
            print(f"   [tune] {params} -> log-loss validation = {ll:.4f}", flush=True)
            if best is None or ll < best[0]:
                best = (ll, params, m, p)
        return best

    val_ll, best_params, m_tr, p_va = select(all_cols, grid, use_ckpt=not a.prune)

    if a.prune:
        drop = []
        for gname, gcols in GROUPS.items():
            gc = [c for c in gcols if c in all_cols]
            if not gc:
                continue
            keep = [c for c in all_cols if c not in gc]
            try:
                ll, _, _, _ = select(keep, [best_params], use_ckpt=False)
                d = ll - val_ll
            except Exception:
                d = 1.0
            print(f"   [prune] bỏ nhóm '{gname}': Δ log-loss validation = {d:+.4f}")
            if d <= 0.0002:
                drop += gc
                pruned.append(gname)
        if drop:
            keep = [c for c in all_cols if c not in drop]
            try:
                ll, params, m, p_new = select(keep, [best_params], use_ckpt=False)
                d = ll - val_ll
            except Exception:
                d = 1.0
            if d <= 0.0005:
                cols, best_params, m_tr, val_ll, p_va = keep, params, m, ll, p_new
                print(f"   -> bỏ các nhóm {pruned} ({len(all_cols) - len(keep)} cột), Δ tổng = {d:+.4f}")
            else:
                pruned = []
                print(f"   -> bỏ đồng thời làm xấu đi ({d:+.4f}), giữ đầy đủ đặc trưng")

    fit_mask = tr if a.pool == "train" else trva
    if a.pool == "train":
        # Mô hình cuối = chính mô hình đã chọn tham số trên validation (ngữ cảnh chỉ từ tập train): ngưỡng, kiểm tra và dự đoán test
        # đều dùng cùng một mô hình; validation không lọt vào ngữ cảnh.
        m_main = m_tr
    else:
        m_main = fit_model(X.loc[trva, cols], y[trva], params=best_params)   # cách cũ: dựng lại từ train+validation
    p_te, sd_te = prob_detail(m_main, X.loc[te, cols], ckpt=None if (a.prune or not use_tabpfn) else ckpt, tag="te")
    res = metrics(y[te], p_te)
    grp = df.loc[te, "lecturer_id"].values if "lecturer_id" in df.columns else None
    ci = boot_auc(y[te], p_te, groups=grp)        # theo cụm giảng viên (đúng hơn)
    ci_rows = boot_auc(y[te], p_te)               # theo từng dòng (để đối chiếu với các bản cũ)

    # ngưỡng trên validation (mô hình chỉ thấy train)
    thr, ok = pick_thresholds(y[va], p_va, max_late_low=a.max_late_low, min_prec_high=a.min_prec_high,
                              safety_margin=a.safety_margin, min_band=a.min_band, cao_top_pct=a.cao_top_pct,
                              safety_margin_high=a.safety_margin_high)
    levels = levels_from(thr)

    print(f"\nMÔ HÌNH CHÍNH: TabPFN {TABPFN_VERSION} | params = {best_params} | log-loss validation = {val_ll:.4f}")
    print("\n=== KẾT QUẢ TRÊN TẬP TEST (theo thời gian) ===\n" + pd.Series({k: round(v, 3) for k, v in res.items()}).to_string())
    print(f"AUC {res['roc_auc']:.3f} (khoảng tin cậy 95%, bootstrap theo giảng viên: {ci[0]:.3f}-{ci[1]:.3f}; "
          f"theo từng dòng: {ci_rows[0]:.3f}-{ci_rows[1]:.3f})")

    lv = pd.DataFrame({"level": [level_of(v, levels) for v in p_te], "y": y[te]}).groupby("level").y.agg(["count", "mean"]).reindex(LEVEL_NAMES)
    lv.columns = ["so_viec", "ti_le_tre_thuc_te"]
    lv["so_viec"] = lv["so_viec"].fillna(0)
    lv["ti_le_viec"] = lv.so_viec / lv.so_viec.sum()
    print(
        f"\n=== MỨC CẢNH BÁO (ngưỡng {thr}"
        f"{' chọn trên validation, có biên an toàn ' + str(a.safety_margin) if ok else ' mặc định [giả định]'}) ===\n"
        + lv.round(3).to_string()
    )
    if ok:
        low_rate, high_rate = lv.loc["Thấp", "ti_le_tre_thuc_te"], lv.loc["Cao", "ti_le_tre_thuc_te"]
        print(f"   Mục tiêu: nhóm Thấp trễ <= {a.max_late_low:.0%} -> thực tế trên test {low_rate:.1%} "
              f"({'đạt' if low_rate <= a.max_late_low else 'KHÔNG đạt'}); nhóm Cao chính xác >= {a.min_prec_high:.0%} -> thực tế {high_rate:.1%} "
              f"({'đạt' if high_rate >= a.min_prec_high else 'KHÔNG đạt'})")

    dec = pd.DataFrame({"p": p_te, "y": y[te]})
    dec["nhom"] = pd.qcut(dec.p, 10, duplicates="drop")
    print("\n=== HIỆU CHUẨN THEO DECILE ===\n" + dec.groupby("nhom", observed=True).mean().round(3).to_string())
    dec_json = [dict(mean_pred=round(float(g.p.mean()), 4), late_rate=round(float(g.y.mean()), 4), n=int(len(g)))
                for _, g in dec.groupby("nhom", observed=True)]

    seg = {}
    Xt = X.loc[te]
    seg_defs = [
        ("giảng viên mới (lịch sử < 5 việc)", (Xt.get("hist_count", pd.Series(99, index=Xt.index)) < 5).values),
        ("giảng viên có lịch sử >= 5 việc", (Xt.get("hist_count", pd.Series(0, index=Xt.index)) >= 5).values),
        ("mùa cao điểm", (Xt.get("peak_season", pd.Series(0, index=Xt.index)) == 1).values),
        ("độ phức tạp cao", (Xt.get("complexity", pd.Series("", index=Xt.index)) == "Cao").values),
    ]
    if "checkpoint_fraction" in Xt.columns:
        seg_defs += [
            ("mốc kiểm tra 25% thời gian (checkpoint 0.25)", (Xt["checkpoint_fraction"] == 0.25).values),
            ("mốc kiểm tra 50% thời gian (checkpoint 0.50)", (Xt["checkpoint_fraction"] == 0.50).values),
        ]
    if "progress_gap" in Xt.columns:
        seg_defs.append(("chậm tiến độ > 10% (progress_gap < -10)", (Xt["progress_gap"].fillna(0) < -10).values))
    if "progress_missing" in Xt.columns:
        seg_defs.append(("chưa cập nhật tiến độ (progress_missing = 1)", (Xt["progress_missing"] == 1).values))
    for sname, mask in seg_defs:
        yy, pp = y[te][mask], p_te[mask]
        seg[sname] = dict(
            n=int(mask.sum()),
            late_rate=round(float(yy.mean()), 3) if mask.sum() else None,
            mean_pred=round(float(pp.mean()), 3) if mask.sum() else None,
            auc=round(float(roc_auc_score(yy, pp)), 3)
            if mask.sum() >= 50 and yy.sum() >= 15 and yy.sum() < len(yy)
            else None,
        )
    print("\n=== HIỆU QUẢ THEO NHÓM ===\n" + pd.DataFrame(seg).T.to_string())

    os.makedirs(a.out, exist_ok=True)
    # --- lưu dự đoán validation/test để báo cáo cuối và so sánh mô hình mà không phải chạy lại GPU ---
    ids = [c for c in ["task_id", "lecturer_id", "faculty", "assigned_date"] if c in df.columns]
    extra = [c for c in ["checkpoint_fraction"] if c in df.columns]
    pd.DataFrame(dict(df.loc[va, ids + extra].reset_index(drop=True).to_dict("list"), y=y[va], risk_probability=np.round(p_va, 5))).to_csv(
        f"{a.out}/predictions_val.csv", index=False, encoding="utf-8-sig")
    pt = pd.DataFrame(dict(df.loc[te, ids + extra].reset_index(drop=True).to_dict("list"), y=y[te], risk_probability=np.round(p_te, 5)))
    if len(m_main["contexts"]) > 1 and use_tabpfn:
        pt["risk_uncertainty"] = np.round(sd_te, 5)
    pt.to_csv(f"{a.out}/predictions_test.csv", index=False, encoding="utf-8-sig")

    meta = make_meta(df, fit_mask, cols, dict(pool=a.pool, n_bags=N_BAGS, max_context=context_limit(), context_mode=CONTEXT_MODE,
                                              tabpfn_version=(TABPFN_VERSION if use_tabpfn else None), no_tune=bool(a.no_tune),
                                              backend=("tabpfn" if use_tabpfn else "hist_gradient_boosting")))
    meta["test_after"] = str(t2.date())
    meta["train_end"] = str(t1.date())
    json.dump(
        dict(
            main_model=("tabpfn" if use_tabpfn else "hist_gradient_boosting"),
            tabpfn_version=(TABPFN_VERSION if use_tabpfn else None),
            device=resolve_device(),
            pool=a.pool,
            context=dict(max_context=context_limit(), n_bags=N_BAGS, mode=CONTEXT_MODE,
                         n_rows_in_context=int(len(m_main["contexts"][0][1]))),
            hyperparams=best_params,
            val_logloss=round(float(val_ll), 4),
            test={k: round(float(v), 4) for k, v in res.items()},
            test_auc_ci95=[round(c, 4) for c in ci],
            test_auc_ci95_method="bootstrap theo cụm giảng viên",
            test_auc_ci95_by_row=[round(c, 4) for c in ci_rows],
            threshold_config=dict(max_late_low=a.max_late_low, min_prec_high=a.min_prec_high, safety_margin=a.safety_margin,
                                  safety_margin_high=a.safety_margin_high, min_band=a.min_band, cao_top_pct=a.cao_top_pct),
            thresholds=thr,
            thresholds_from_target_precision=ok,
            pruned_groups=pruned,
            segments=seg,
            alert_levels={k: dict(n=int(r.so_viec), late_rate=(None if r.so_viec == 0 else round(float(r.ti_le_tre_thuc_te), 4)),
                                  share=round(float(r.ti_le_viec), 4)) for k, r in lv.iterrows()},
            target_check=(dict(max_late_low=a.max_late_low, actual_low=_nn(lv.loc["Thấp", "ti_le_tre_thuc_te"]),
                               ok_low=bool(lv.loc["Thấp", "ti_le_tre_thuc_te"] <= a.max_late_low),
                               min_prec_high=a.min_prec_high, actual_high=_nn(lv.loc["Cao", "ti_le_tre_thuc_te"]),
                               ok_high=bool(lv.loc["Cao", "ti_le_tre_thuc_te"] >= a.min_prec_high)) if ok else None),
            calibration_deciles=dec_json,
            n_train=int(fit_mask.sum()),
            n_test=int(te.sum()),
            meta=meta,
        ),
        open(f"{a.out}/metrics.json", "w", encoding="utf-8"),
        ensure_ascii=False,
        indent=2,
    )
    base = baseline_values(X.loc[fit_mask, cols])
    out = slim(m_main)
    out.update(levels=levels, thresholds=thr, baseline=base, meta=meta)
    joblib.dump(out, f"{a.out}/risk_model.joblib", compress=3)
    print(f"\nĐã lưu ngữ cảnh + bộ mã hóa + báo cáo vào {os.path.abspath(a.out)}")
    if not a.keep_checkpoint:
        import shutil
        shutil.rmtree(ckpt, ignore_errors=True)
    if a.importance_rows > 0:
        print(f"\nĐang tính độ quan trọng đặc trưng (hoán vị, {a.importance_rows} dòng validation; mô hình ĐÃ được lưu, có thể dừng bước này)...", flush=True)
        imp = permutation_importance(m_tr, X.loc[va, cols], y[va], n_rows=a.importance_rows)
        print("=== TOP 12 ĐẶC TRƯNG (giảm AUC khi xáo trộn) ===\n" + imp.head(12).round(4).to_string(index=False))
        imp.to_csv(f"{a.out}/feature_importance.csv", index=False)

    ex = df[te].assign(p=p_te).sort_values("p", ascending=False).head(3)
    texts = explain(dict(m_main, baseline=base), X.loc[ex.index, cols])
    print("\n=== VÍ DỤ CẢNH BÁO ===")
    for (_, r), tx in zip(ex.iterrows(), texts):
        print(
            f"- {r.task_id} ({r.task_type if 'task_type' in r else ''}): có {r.p:.0%} khả năng giảng viên {r.lecturer_id} "
            f"trễ deadline này -> mức {level_of(r.p, levels)} "
            f"(thực tế: {'trễ' if r[LABEL] else 'đúng hạn'})\n    {tx}"
        )


STALE_DAYS = 180          # cảnh báo nếu dữ liệu cần dự đoán cách dữ liệu huấn luyện quá số ngày này


def predict(a):
    import time
    t0 = time.time()
    df = pd.read_csv(a.input)
    eng = engineer(df)
    model_path = a.model
    # Tự động chuyển sang model_dynamic/risk_model.joblib nếu dữ liệu đầu vào có cột tiến độ (%) và người dùng đang trỏ tới model tĩnh mặc định
    if ("progress_pct" in eng.columns or "checkpoint_fraction" in eng.columns):
        m_try = joblib.load(model_path)
        if "progress_pct" not in m_try.get("features", []):
            dyn_cand = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(model_path))), "model_dynamic", "risk_model.joblib")
            if os.path.exists(dyn_cand):
                print(f"[TỰ ĐỘNG] Phát hiện dữ liệu có tiến độ hoàn thành (%) -> chuyển sang Mô hình Động: {dyn_cand}")
                model_path = dyn_cand
                m = joblib.load(model_path)
            else:
                m = m_try
        else:
            m = m_try
    else:
        m = joblib.load(model_path)
    # dùng đúng phiên bản/thiết bị lúc huấn luyện, trừ khi người dùng ghi đè
    global TABPFN_VERSION
    if not a.tabpfn_version:
        TABPFN_VERSION = m.get("tabpfn_version", TABPFN_VERSION)
    eng = restore_dups(eng, m["features"])
    meta = m.get("meta")
    is_dyn = "progress_pct" in m.get("features", [])
    model_tag = "MÔ HÌNH ĐỘNG (cập nhật theo % tiến độ)" if is_dyn else "MÔ HÌNH TĨNH (lúc mới giao việc)"
    if TabPFNClassifier is None:
        raise SystemExit("[DỪNG] Chưa cài tabpfn: mô hình này chạy bằng TabPFN (pip install tabpfn; GPU được khuyến nghị).")
    mode_str = f"{model_tag} | TabPFN {m.get('tabpfn_version', '?')} ({len(m['contexts'])} túi)"
    if meta:
        print(f"Mô hình: {mode_str} | huấn luyện {meta['trained_at'][:10]} trên dữ liệu "
              f"{meta['data_start']} -> {meta['data_end']} ({meta['n_rows']:,} dòng)")
    lost = [c for c in m["features"] if c not in eng.columns]
    if lost:
        print(
            f"[CẢNH BÁO] Thiếu {len(lost)}/{len(m['features'])} cột đặc trưng (sẽ coi là trống): {lost[:8]}...\n"
            "          -> chạy clean_data.py --inference (và build_task_personnel_matrix nếu cần) trước khi dự đoán."
        )
    X = eng.reindex(columns=m["features"])

    # --- kiểm tra đầu vào ---
    unseen = unseen_categories(m["encoder"], X)
    for c, info in unseen.items():
        print(f"[CẢNH BÁO] Cột '{c}': {info['n']} giá trị ({info['pct']}%) chưa từng thấy lúc huấn luyện, ví dụ {info['examples']} "
              "-> được coi là thiếu; dự đoán cho các dòng này kém tin cậy.")
    if meta and "assigned_date" in df.columns:
        try:
            gap = (pd.to_datetime(df["assigned_date"]).max() - pd.Timestamp(meta["data_end"])).days
            if gap > STALE_DAYS:
                print(f"[CẢNH BÁO] Dữ liệu cần dự đoán muộn hơn dữ liệu huấn luyện {gap} ngày (> {STALE_DAYS}); "
                      "nên kiểm tra hiệu chuẩn trên dữ liệu gần đây hoặc huấn luyện lại.")
        except Exception:
            pass

    t_pred = time.time()
    p, sd = prob_detail(m, X)
    dyn_cols = [c for c in ["checkpoint_fraction", "days_elapsed", "days_remaining", "progress_pct", "expected_progress_pct", "progress_gap", "progress_missing"] if c in eng.columns]
    out = eng[[c for c in ID_COLS + dyn_cols if c in eng.columns]].copy()
    out["risk_probability"] = p.round(3)
    out["risk_level"] = [level_of(v, m["levels"]) for v in p]
    if len(m["contexts"]) > 1:
        out["risk_uncertainty"] = sd.round(3)      # độ lệch chuẩn giữa các túi: càng lớn mô hình càng không chắc
    notes = [[] for _ in range(len(X))]
    hc = X["hist_count"] if "hist_count" in X.columns else pd.Series(np.nan, index=X.index)
    cold = (hc.fillna(0) < 5).values
    if "is_cold_start" in X.columns:
        cold = cold | (X["is_cold_start"].fillna(0) == 1).values
    for i in np.where(cold)[0]:
        notes[i].append("Giảng viên mới (lịch sử < 5 việc): độ tin cậy thấp")
    for c in unseen:
        v = X[c].astype(str)
        bad = (~v.isin(m["encoder"]["maps"][c].keys())) & X[c].notna().values
        for i in np.where(bad.values)[0]:
            notes[i].append(f"'{c}' có giá trị chưa từng thấy")
    if "progress_missing" in X.columns:
        pmiss = (X["progress_missing"].fillna(0) == 1).values
        for i in np.where(pmiss)[0]:
            notes[i].append("Chưa cập nhật tiến độ: nguy cơ trễ hạn ẩn")
    if "progress_gap" in X.columns:
        pgap = X["progress_gap"].fillna(0).values
        for i in np.where(pgap < -10.0)[0]:
            notes[i].append(f"Chậm tiến độ {abs(pgap[i]):.0f}% so với kỳ vọng")
    out["ghi_chu"] = ["; ".join(x) for x in notes]
    if a.explain:
        out["top_factors"] = explain(m, X)
    infer_ms = (time.time() - t_pred) * 1000
    out.to_csv(a.output, index=False, encoding="utf-8-sig")
    print(out.drop(columns=["top_factors"], errors="ignore").head(10).to_string(index=False))
    print(f"\nĐã ghi {a.output} ({len(out)} dòng | thời gian suy luận{' + giải thích' if a.explain else ''}: {infer_ms:.1f} ms)")


def configure(a):
    """Nhận cấu hình TabPFN từ dòng lệnh."""
    global TABPFN_VERSION, DEVICE, MAX_CONTEXT, N_BAGS, CONTEXT_MODE, FIT_MODE
    if getattr(a, "tabpfn_version", None):
        TABPFN_VERSION = a.tabpfn_version
    DEVICE = getattr(a, "device", "auto") or "auto"
    MAX_CONTEXT = getattr(a, "max_context", None)
    N_BAGS = getattr(a, "n_bags", 1) or 1
    CONTEXT_MODE = getattr(a, "context_mode", "random") or "random"
    FIT_MODE = getattr(a, "fit_mode", None) or None
    if getattr(a, "cpu_light", False):
        # Chế độ rút gọn ngữ cảnh TabPFN xuống 1 túi x 2048 dòng khi chạy mạng TabPFN trên CPU (chưa đánh giá chất lượng)
        DEVICE = "cpu"
        MAX_CONTEXT = CPU_LIGHT_CONTEXT_CAP
        N_BAGS = 1


def add_tabpfn_args(p):
    p.add_argument("--tabpfn-version", default=None, help="v2 (mặc định, mở) | v2.5 | v2.6 ... (cần HF token + giấy phép)")
    p.add_argument("--device", default="auto", choices=["auto", "cpu", "cuda"])
    p.add_argument("--max-context", type=int, default=None, help="số dòng ngữ cảnh tối đa/túi (mặc định 10000; 50000 nếu v2.5+ trên GPU)")
    p.add_argument("--n-bags", type=int, default=1, help="số túi ngữ cảnh độc lập, lấy trung bình xác suất")
    p.add_argument("--context-mode", default="random", choices=["random", "recent"])
    p.add_argument("--fit-mode", default=None, choices=["low_memory", "fit_preprocessors", "fit_with_cache"], help="chế độ lưu đệm của TabPFN; fit_with_cache nhanh hơn nhưng tốn bộ nhớ GPU (thử ở MODE quick trước)")
    p.add_argument("--cpu-light", action="store_true",
                   help="chạy TabPFN trên CPU với ngữ cảnh rút gọn (1 túi x 2.048 dòng gần nhất); chất lượng chưa được đánh giá")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    sp = ap.add_subparsers(dest="cmd", required=True)
    t = sp.add_parser("train")
    t.add_argument("--data", required=True)
    t.add_argument("--out", default="./model")
    t.add_argument("--prune", action="store_true", help="loại các nhóm đặc trưng không giúp ích trên validation (chậm)")
    t.add_argument("--pool", default="train", choices=["train", "trainval"], help="nguồn ngữ cảnh của mô hình cuối: train (mặc định, nhất quán với ngưỡng chọn trên validation) hoặc trainval")
    t.add_argument("--resume", action="store_true", help="tiếp tục từ điểm lưu trong <out>/checkpoint nếu cấu hình và dữ liệu không đổi")
    t.add_argument("--keep-checkpoint", action="store_true", help="giữ lại thư mục điểm lưu sau khi huấn luyện xong")
    t.add_argument("--no-tune", action="store_true", help="bỏ chọn siêu tham số, dùng cấu hình đầu tiên (nhanh)")
    t.add_argument("--importance-rows", type=int, default=0, help="số dòng tính độ quan trọng hoán vị; 0 = bỏ qua (mặc định). RẤT TỐN: ~ (số đặc trưng+1) x số túi x số dòng lượt suy luận")
    t.add_argument("--max-late-low", type=float, default=0.10, help="mục tiêu: nhóm 'Thấp' có tỉ lệ trễ tối đa")
    t.add_argument("--min-prec-high", type=float, default=0.60, help="mục tiêu: nhóm 'Cao' có độ chính xác tối thiểu")
    t.add_argument("--safety-margin", type=float, default=0.01, help="biên an toàn khi chọn ngưỡng (đề phòng lệch giữa validation và giai đoạn sau)")
    t.add_argument("--safety-margin-high", type=float, default=None, help="biên an toàn RIÊNG cho mức Cao (mặc định = --safety-margin)")
    t.add_argument("--min-band", type=float, default=0.05, help="độ rộng tối thiểu của dải 'Trung bình'")
    t.add_argument("--cao-top-pct", type=float, default=None, help="(tuỳ chọn) đặt ngưỡng 'Cao' để đúng top x%% việc rơi vào nhóm Cao theo năng lực xử lý")
    add_tabpfn_args(t)
    t.set_defaults(f=train)
    q = sp.add_parser("predict")
    q.add_argument("--model", required=True)
    q.add_argument("--input", required=True)
    q.add_argument("--output", default="predictions.csv")
    q.add_argument("--explain", action="store_true")
    add_tabpfn_args(q)
    q.set_defaults(f=predict)
    a = ap.parse_args()
    configure(a)
    a.f(a)


In [ ]:
%%writefile run_baselines.py
#!/usr/bin/env python3
"""
CÁC MÔ HÌNH ĐỐI CHỨNG (CPU, không cần GPU), cùng quy trình với train_model.py (--pool train):
  huấn luyện trên tập train, chọn siêu tham số trên validation, dự đoán validation và test.
Mô hình: baseline lịch sử, hồi quy logistic, hồi quy logistic có spline, CatBoost
(và với dữ liệu ĐỘNG: thêm CatBoost chỉ dùng đặc trưng tĩnh để đo giá trị của thông tin tiến độ).

  python run_baselines.py --data ./data/ml_training_set_with_matrix.csv --out ./baselines_static
  python run_baselines.py --data ./data/ml_dynamic_set_clean_with_matrix.csv --out ./baselines_dynamic

Ra: predictions_val.csv, predictions_test.csv (mỗi mô hình một cột), baselines_info.json
"""
import argparse, json, os, time, warnings
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, SplineTransformer, StandardScaler
warnings.filterwarnings("ignore")
import train_model as T

try:
    from catboost import CatBoostClassifier
except ImportError:
    CatBoostClassifier = None
CB_GRID = [dict(depth=6, learning_rate=0.05, l2_leaf_reg=3.0, iterations=400), dict(depth=4, learning_rate=0.03, l2_leaf_reg=5.0, iterations=500),
           dict(depth=8, learning_rate=0.05, l2_leaf_reg=1.0, iterations=300), dict(depth=6, learning_rate=0.08, l2_leaf_reg=3.0, iterations=250)]
DYN_COLS = ["checkpoint_fraction", "days_elapsed", "days_remaining", "progress_pct", "expected_progress_pct", "progress_gap", "progress_missing"]
clip = lambda p: np.clip(p, 1e-4, 1 - 1e-4)


def make_lr(X, C, spline=False, n_knots=5):
    cat, num = T.split_cols(X)
    if spline:
        cont = [c for c in num if X[c].nunique() > 10]
        low = [c for c in num if c not in cont]
        parts = [("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=30), cat),
                 ("low", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), low),
                 ("cont", make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), SplineTransformer(n_knots=n_knots, degree=3, extrapolation="constant")), cont)]
    else:
        parts = [("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=30), cat),
                 ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num)]
    return Pipeline([("pre", ColumnTransformer(parts)), ("m", LogisticRegression(C=C, max_iter=3000))])


def cb_prep(X):
    X = X.copy()
    cat, num = T.split_cols(X)
    for c in cat:
        X[c] = X[c].astype(object).where(X[c].notna(), None).map(lambda v: str(v) if v is not None and v == v else None)
    for c in num:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    return X, cat


def cb_fit(X, y, params):
    Xp, cat = cb_prep(X)
    m = CatBoostClassifier(loss_function="Logloss", random_seed=0, verbose=False, allow_writing_files=False, cat_features=cat, **params)
    m.fit(Xp, y)
    return m


def cb_prob(m, X):
    return clip(m.predict_proba(cb_prep(X)[0])[:, 1])


def tune(name, fits, X, y, tr, va, info):
    """fits: danh sách (tham số, hàm dựng mô hình đã fit, hàm dự đoán). Chọn theo log-loss validation."""
    best = None
    for params, fit, pred in fits:
        t = time.time()
        m = fit(X.loc[tr], y[tr])
        p = pred(m, X.loc[va])
        ll = log_loss(y[va], p)
        print(f"   [{name}] {params} -> log-loss validation {ll:.4f} ({time.time() - t:.0f}s)", flush=True)
        if best is None or ll < best[0]:
            best = (ll, params, m, pred)
    info[name] = dict(val_logloss=round(float(best[0]), 5), params=best[1])
    return best[2], best[3]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", default="./baselines")
    ap.add_argument("--no-catboost", action="store_true")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    df = T.engineer(pd.read_csv(a.data, parse_dates=["assigned_date"])).sort_values("assigned_date").reset_index(drop=True)
    y = df[T.LABEL].astype(int).values
    X = df.drop(columns=[c for c in T.ID_COLS + T.TARGETS if c in df.columns])
    t1, t2 = df.assigned_date.quantile(0.60), df.assigned_date.quantile(0.75)
    tr = (df.assigned_date <= t1).values
    va = ((df.assigned_date > t1) & (df.assigned_date <= t2)).values
    te = (df.assigned_date > t2).values
    dynamic = "checkpoint_fraction" in df.columns
    print(f"{'ĐỘNG' if dynamic else 'TĨNH'} | train {tr.sum()} | validation {va.sum()} | test {te.sum()} | {X.shape[1]} đặc trưng", flush=True)
    ids = [c for c in ["task_id", "lecturer_id", "faculty", "assigned_date", "checkpoint_fraction"] if c in df.columns]
    out_va = df.loc[va, ids].reset_index(drop=True).assign(y=y[va])
    out_te = df.loc[te, ids].reset_index(drop=True).assign(y=y[te])
    info = {}
    models = {}

    out_va["hist_baseline"] = df.loc[va, "hist_late_rate"].fillna(0.2).values
    out_te["hist_baseline"] = df.loc[te, "hist_late_rate"].fillna(0.2).values
    models["logistic_base"] = (X, [(dict(C=c), (lambda Xa, ya, c=c: make_lr(Xa, c).fit(Xa, ya)), lambda m, Xa: clip(m.predict_proba(Xa)[:, 1])) for c in [0.01, 0.05, 0.1, 0.3, 1.0]])
    models["logistic_spline"] = (X, [(dict(C=c), (lambda Xa, ya, c=c: make_lr(Xa, c, spline=True).fit(Xa, ya)), lambda m, Xa: clip(m.predict_proba(Xa)[:, 1])) for c in [0.03, 0.1, 0.3, 1.0]])
    if CatBoostClassifier is not None and not a.no_catboost:
        models["catboost"] = (X, [(p, (lambda Xa, ya, p=p: cb_fit(Xa, ya, p)), cb_prob) for p in CB_GRID])
        if dynamic:
            Xs = X.drop(columns=[c for c in DYN_COLS if c in X.columns])
            models["catboost_khong_tien_do"] = (Xs, [(p, (lambda Xa, ya, p=p: cb_fit(Xa, ya, p)), cb_prob) for p in CB_GRID])
    else:
        print("(bỏ qua CatBoost: chưa cài hoặc --no-catboost)")
    for name, (XX, fits) in models.items():
        m, pred = tune(name, fits, XX, y, tr, va, info)
        out_va[name] = pred(m, XX.loc[va])
        out_te[name] = pred(m, XX.loc[te])
        r = T.metrics(y[te], out_te[name].values)
        print(f"== {name}: test AUC {r['roc_auc']:.4f} | PR-AUC {r['pr_auc']:.4f} | Brier {r['brier']:.4f} | ECE {r['ece']:.4f}", flush=True)
        info[name]["test"] = {k: round(float(v), 4) for k, v in r.items()}
        out_va.to_csv(f"{a.out}/predictions_val.csv", index=False, encoding="utf-8-sig")
        out_te.to_csv(f"{a.out}/predictions_test.csv", index=False, encoding="utf-8-sig")
    json.dump(dict(dynamic=dynamic, n_train=int(tr.sum()), n_val=int(va.sum()), n_test=int(te.sum()), models=info), open(f"{a.out}/baselines_info.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(f"\nĐã lưu vào {os.path.abspath(a.out)}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile final_report.py
#!/usr/bin/env python3
"""
BÁO CÁO CUỐI: ghép dự đoán TabPFN (từ train_model.py) với các mô hình đối chứng (run_baselines.py) trên CÙNG tập test,
tính chỉ số, khoảng tin cậy bootstrap THEO CỤM GIẢNG VIÊN, so sánh cặp, hiệu chuẩn theo quý, nhóm con, so với xác suất thật (nếu có),
rồi xuất bảng, hình và BAO_CAO.md (không cần GPU).

  python final_report.py --tabpfn ./model --baselines ./baselines_static --data ./data/ml_training_set_with_matrix.csv \\
      --oracle ./data/oracle_truth.csv --out ./bao_cao_tinh --title "Mô hình tĩnh"
"""
import argparse, json, os, warnings
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
warnings.filterwarnings("ignore")
import train_model as T

NAMES = {"tabpfn": "TabPFN v2", "catboost": "CatBoost", "logistic_spline": "Logistic (spline)", "logistic_base": "Logistic (tuyến tính)",
         "hist_baseline": "Baseline lịch sử", "catboost_khong_tien_do": "CatBoost (bỏ đặc trưng tiến độ)"}
clip = lambda p: np.clip(p, 1e-4, 1 - 1e-4)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--tabpfn", required=True, help="thư mục model có predictions_test.csv và metrics.json")
    ap.add_argument("--baselines", required=True, help="thư mục run_baselines.py")
    ap.add_argument("--data", default=None, help="file dữ liệu (để lấy thuộc tính nhóm con)")
    ap.add_argument("--oracle", default=None, help="oracle_truth.csv (chỉ dùng cho dữ liệu tĩnh)")
    ap.add_argument("--out", default="./bao_cao")
    ap.add_argument("--title", default="Mô hình")
    ap.add_argument("--boot", type=int, default=400)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    tab = pd.read_csv(f"{a.tabpfn}/predictions_test.csv", parse_dates=["assigned_date"])
    bas = pd.read_csv(f"{a.baselines}/predictions_test.csv", parse_dates=["assigned_date"])
    keys = [c for c in ["task_id", "lecturer_id", "checkpoint_fraction"] if c in tab.columns and c in bas.columns]
    bcols = [c for c in bas.columns if c not in ["task_id", "lecturer_id", "faculty", "assigned_date", "checkpoint_fraction", "y"]]
    m = tab.rename(columns={"risk_probability": "tabpfn"}).merge(bas[keys + bcols + ["y"]], on=keys, suffixes=("", "_b"))
    assert len(m) == len(tab) == len(bas), f"không ghép đủ dòng: {len(m)} / {len(tab)} / {len(bas)}"
    assert (m.y == m.y_b).all(), "nhãn hai nguồn không khớp"
    y = m.y.values.astype(int)
    models = ["tabpfn"] + bcols
    dynamic = "checkpoint_fraction" in m.columns
    meta = json.load(open(f"{a.tabpfn}/metrics.json", encoding="utf-8")) if os.path.exists(f"{a.tabpfn}/metrics.json") else {}
    P = {k: m[k].values for k in models}
    print(f"{a.title} ({'động' if dynamic else 'tĩnh'}): {len(m):,} dòng test, tỉ lệ trễ {y.mean():.4f}; mô hình: {models}")

    # --- 1. chỉ số ---
    rows = []
    for k in models:
        r = T.metrics(y, P[k]); r["logloss"] = log_loss(y, clip(P[k])); r["model"] = k; rows.append(r)
    M = pd.DataFrame(rows).set_index("model")[["roc_auc", "pr_auc", "brier", "logloss", "ece", "recall_at_top20pct", "precision_at_top20pct"]]

    # --- 2. bootstrap theo cụm giảng viên (cặp) ---
    grp = m.lecturer_id.values
    order = np.argsort(grp, kind="stable")
    ug, st = np.unique(grp[order], return_index=True)
    bnd = list(st) + [len(grp)]
    members = [order[bnd[j]:bnd[j + 1]] for j in range(len(ug))]
    rng = np.random.default_rng(0)
    R = {k: dict(a=[], b=[], l=[]) for k in models}
    for _ in range(a.boot):
        i = np.concatenate([members[j] for j in rng.integers(0, len(ug), len(ug))])
        yy = y[i]
        if yy.min() == yy.max():
            continue
        for k in models:
            p = P[k][i]
            R[k]["a"].append(roc_auc_score(yy, p)); R[k]["b"].append(brier_score_loss(yy, p)); R[k]["l"].append(log_loss(yy, clip(p)))
    ci = lambda v: [round(float(np.percentile(v, 2.5)), 4), round(float(np.percentile(v, 97.5)), 4)]
    paired = {}
    for k in models[1:]:
        da, db, dl = [np.array(R["tabpfn"][x]) - np.array(R[k][x]) for x in "abl"]
        paired[k] = dict(d_auc=round(float(da.mean()), 4), d_auc_ci=ci(da), d_brier=round(float(db.mean()), 5), d_brier_ci=[round(float(np.percentile(db, 2.5)), 5), round(float(np.percentile(db, 97.5)), 5)],
                         d_logloss=round(float(dl.mean()), 5))

    # --- 3. nhóm con ---
    seg = {}
    if a.data and os.path.exists(a.data):
        at = pd.read_csv(a.data, usecols=lambda c: c in keys + ["hist_count", "complexity", "peak_season"])
        mm = m.merge(at, on=keys, how="left")
        masks = {"giảng viên mới (<5 việc)": (mm.hist_count < 5).values, "giảng viên có lịch sử (>=5)": (mm.hist_count >= 5).values,
                 "mùa cao điểm": (mm.peak_season == 1).values, "độ phức tạp cao": (mm.complexity == "Cao").values}
        for f in sorted(m.faculty.dropna().unique()):
            masks[f"Khoa {f}"] = (m.faculty == f).values
        if dynamic:
            for fr in sorted(m.checkpoint_fraction.unique()):
                masks[f"mốc tiến độ {fr:.0%}"] = (m.checkpoint_fraction == fr).values
        for sn, mk in masks.items():
            seg[sn] = dict(n=int(mk.sum()), **{k: (round(float(roc_auc_score(y[mk], P[k][mk])), 3) if mk.sum() >= 50 and 15 <= y[mk].sum() < mk.sum() else None) for k in models})
    S = pd.DataFrame(seg).T if seg else pd.DataFrame()

    # --- 4. hiệu chuẩn theo quý ---
    q = m.assigned_date.dt.to_period("Q").astype(str)
    cal = []
    for qq in sorted(q.unique()):
        mk = (q == qq).values
        row = dict(quy=qq, n=int(mk.sum()), ti_le_tre_that=round(float(y[mk].mean()), 4))
        for k in models:
            row[f"{k}_dd_TB"] = round(float(P[k][mk].mean()), 4)
        cal.append(row)
    C = pd.DataFrame(cal)

    # --- 5. so với xác suất thật ---
    O = None
    if a.oracle and os.path.exists(a.oracle) and not dynamic:
        orc = pd.read_csv(a.oracle)[["task_id", "lecturer_id", "p_true_observed"]]
        mo = m.merge(orc, on=["task_id", "lecturer_id"], how="left")
        po = mo.p_true_observed.values
        auc_o = roc_auc_score(y, po)
        O = pd.DataFrame([dict(model=k, auc=roc_auc_score(y, P[k]), khoang_cach_toi_oracle=auc_o - roc_auc_score(y, P[k]), pearson=np.corrcoef(P[k], po)[0, 1], dd_TB=P[k].mean()) for k in models]).set_index("model")
        O.attrs["auc_oracle"] = auc_o; O.attrs["mean_oracle"] = po.mean()

    # --- 6. hình ---
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        from sklearn.metrics import roc_curve
        plt.rcParams.update({"font.size": 11, "axes.grid": True, "grid.alpha": .3})
        col = {"tabpfn": "#d62728", "catboost": "#1f77b4", "logistic_spline": "#2ca02c", "logistic_base": "#9467bd", "hist_baseline": "#7f7f7f", "catboost_khong_tien_do": "#17becf"}
        fig, ax = plt.subplots(figsize=(6, 5.2))
        for k in models:
            f, t, _ = roc_curve(y, P[k]); ax.plot(f, t, color=col.get(k), lw=2 if k == "tabpfn" else 1.3, label=f"{NAMES.get(k, k)} (AUC {M.loc[k, 'roc_auc']:.3f})")
        ax.plot([0, 1], [0, 1], "k:"); ax.set_xlabel("Tỉ lệ dương tính giả"); ax.set_ylabel("Tỉ lệ dương tính thật"); ax.set_title(f"ROC - {a.title}"); ax.legend(loc="lower right", fontsize=8)
        fig.tight_layout(); fig.savefig(f"{a.out}/hinh_ROC.png", dpi=160); plt.close(fig)
        fig, ax = plt.subplots(figsize=(6, 5.2))
        for k in [x for x in models if x != "hist_baseline"]:
            qq = pd.qcut(P[k], 10, labels=False, duplicates="drop"); g = pd.DataFrame({"p": P[k], "y": y}).groupby(qq).mean()
            ax.plot(g.p, g.y, "o-", ms=3, lw=2 if k == "tabpfn" else 1.2, color=col.get(k), label=f"{NAMES.get(k, k)} (ECE {M.loc[k, 'ece']:.3f})")
        ax.plot([0, .8], [0, .8], "k:"); ax.set_xlabel("Xác suất dự đoán TB (decile)"); ax.set_ylabel("Tỉ lệ trễ thực tế"); ax.set_title(f"Hiệu chuẩn - {a.title}"); ax.legend(fontsize=8)
        fig.tight_layout(); fig.savefig(f"{a.out}/hinh_hieu_chuan.png", dpi=160); plt.close(fig)
        fig, ax = plt.subplots(figsize=(7, 0.55 * len(models) + 1.6))
        av = [M.loc[k, "roc_auc"] for k in models]; lo = [ci(R[k]["a"])[0] for k in models]; hi = [ci(R[k]["a"])[1] for k in models]
        ax.barh([NAMES.get(k, k) for k in models][::-1], av[::-1], xerr=[np.array(av[::-1]) - np.array(lo[::-1]), np.array(hi[::-1]) - np.array(av[::-1])], color=[col.get(k) for k in models][::-1], capsize=3, alpha=.85)
        ax.set_xlim(max(.5, min(lo) - .03), min(.9, max(hi) + .02)); ax.set_xlabel("AUC (KTC 95%, bootstrap theo giảng viên)"); ax.set_title(f"So sánh AUC - {a.title}")
        fig.tight_layout(); fig.savefig(f"{a.out}/hinh_AUC.png", dpi=160); plt.close(fig)
        fig, ax = plt.subplots(figsize=(7.5, 4.2))
        ax.plot(C.quy, C.ti_le_tre_that, "k-o", label="Thực tế")
        for k in ["tabpfn"] + [x for x in ["catboost", "logistic_spline"] if x in models]:
            ax.plot(C.quy, C[f"{k}_dd_TB"], "--s", ms=3, color=col.get(k), label=NAMES.get(k, k))
        ax.set_ylabel("Tỉ lệ trễ trung bình"); ax.set_title(f"Hiệu chuẩn theo quý (tập test) - {a.title}"); ax.legend(fontsize=8); plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
        fig.tight_layout(); fig.savefig(f"{a.out}/hinh_hieu_chuan_theo_quy.png", dpi=160); plt.close(fig)
    except Exception as e:
        print("(bỏ qua vẽ hình:", e, ")")

    # --- 7. xuất ---
    M.round(5).to_csv(f"{a.out}/bang_chi_so.csv"); C.to_csv(f"{a.out}/bang_hieu_chuan_theo_quy.csv", index=False)
    if len(S): S.to_csv(f"{a.out}/bang_auc_theo_nhom.csv")
    if O is not None: O.round(5).to_csv(f"{a.out}/bang_so_voi_oracle.csv")
    out = dict(title=a.title, dynamic=dynamic, n_test=int(len(m)), late_rate=round(float(y.mean()), 4), metrics=M.round(5).to_dict("index"),
               auc_ci95={k: ci(R[k]["a"]) for k in models}, brier_ci95={k: [round(float(np.percentile(R[k]["b"], 2.5)), 5), round(float(np.percentile(R[k]["b"], 97.5)), 5)] for k in models},
               paired_tabpfn_minus=paired, segments_auc=S.to_dict("index") if len(S) else {})
    json.dump(out, open(f"{a.out}/so_sanh_cap.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=float)
    f = lambda v: f"{v:.4f}"
    md = [f"# Báo cáo cuối: {a.title}", "", f"Tập test: **{len(m):,}** dòng, tỉ lệ trễ **{y.mean():.2%}**. Khoảng tin cậy 95% bootstrap theo cụm giảng viên ({a.boot} lần).", "",
          "## 1. Chỉ số trên tập test", "", "| Mô hình | AUC [KTC 95%] | PR-AUC | Brier | Log-loss | ECE |", "|---|---|---|---|---|---|"]
    for k in models:
        md.append(f"| {NAMES.get(k, k)} | {f(M.loc[k, 'roc_auc'])} [{ci(R[k]['a'])[0]:.3f}; {ci(R[k]['a'])[1]:.3f}] | {f(M.loc[k, 'pr_auc'])} | {f(M.loc[k, 'brier'])} | {f(M.loc[k, 'logloss'])} | {f(M.loc[k, 'ece'])} |")
    md += ["", "## 2. So sánh cặp: TabPFN trừ mô hình khác", "", "| So với | ΔAUC [KTC 95%] | ΔBrier | Δlog-loss |", "|---|---|---|---|"]
    for k, p in paired.items():
        md.append(f"| {NAMES.get(k, k)} | {p['d_auc']:+.4f} [{p['d_auc_ci'][0]:+.4f}; {p['d_auc_ci'][1]:+.4f}] | {p['d_brier']:+.5f} | {p['d_logloss']:+.5f} |")
    md += ["", "(ΔAUC > 0: TabPFN tốt hơn; ΔBrier, Δlog-loss < 0: TabPFN tốt hơn. Khoảng tin cậy chứa 0: chưa đủ bằng chứng về khác biệt.)"]
    if meta.get("alert_levels"):
        md += ["", "## 3. Mức cảnh báo của TabPFN (ngưỡng chọn trên validation)", "", f"Ngưỡng: {meta.get('thresholds')} | mục tiêu: {meta.get('target_check')}", "", "| Mức | Số việc | % số việc | Tỉ lệ trễ thực tế |", "|---|---|---|---|"]
        for k, v in meta["alert_levels"].items():
            md.append(f"| {k} | {v['n']:,} | {v['share']:.1%} | {('-' if v['late_rate'] is None else format(v['late_rate'], '.2%'))} |")
    md += ["", "## 4. Hiệu chuẩn theo quý (tỉ lệ trễ thực tế và dự đoán trung bình)", "", C.to_markdown(index=False) if hasattr(C, "to_markdown") else C.to_string(index=False)]
    try:
        if len(S):
            md += ["", "## 5. AUC theo nhóm", "", S.to_markdown() if hasattr(S, "to_markdown") else S.to_string()]
    except Exception:
        md += ["", S.to_string()]
    if O is not None:
        md += ["", f"## 6. So với xác suất 'thật' của bộ sinh dữ liệu (oracle AUC {O.attrs['auc_oracle']:.4f}, TB {O.attrs['mean_oracle']:.3f})", "", O.round(4).to_string()]
    md += ["", "## Lưu ý", "- Dữ liệu mô phỏng: chỉ số phản ánh khả năng học lại quy luật đã mô phỏng, không phải hiệu quả trên dữ liệu thật.",
           "- Mọi lựa chọn (cấu hình, ngưỡng) chốt trên validation; tập test chỉ dùng để báo cáo."]
    open(f"{a.out}/BAO_CAO.md", "w", encoding="utf-8").write("\n".join(md))
    print(M.round(4).to_string()); print(f"\nĐã lưu bảng, hình và BAO_CAO.md vào {os.path.abspath(a.out)}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile report_levels.py
#!/usr/bin/env python3
"""
BÁO CÁO MỨC CẢNH BÁO TỪ MÔ HÌNH ĐÃ LƯU (không huấn luyện lại).
Nạp model/risk_model.joblib, dự đoán trên TẬP TEST (cùng cách chia theo thời gian 60/15/25 như train_model.py),
rồi in và lưu: bảng mức cảnh báo, kiểm tra mục tiêu (đạt/không đạt), hiệu chuẩn theo decile, hiệu quả theo nhóm,
và dự đoán từng việc để phân tích tiếp (không phải chạy lại GPU).

  python report_levels.py --data ./data/ml_training_set_with_matrix.csv --model ./model/risk_model.joblib --out ./model_report

Chi phí chủ yếu là suy luận trên ~25.000 dòng test x số túi (xem 'n_bags' của mô hình); nên dùng GPU.
Cho phép dừng giữa chừng và chạy tiếp (lô đã xong được lưu trong <out>/checkpoint_report).
"""
import argparse, json, os, shutil
import numpy as np, pandas as pd, joblib
from sklearn.metrics import roc_auc_score
import train_model as T

ap = argparse.ArgumentParser()
ap.add_argument("--data", required=True)
ap.add_argument("--model", required=True)
ap.add_argument("--out", default="./model_report")
ap.add_argument("--max-late-low", type=float, default=0.10)
ap.add_argument("--min-prec-high", type=float, default=0.60)
ap.add_argument("--keep-checkpoint", action="store_true")
T.add_tabpfn_args(ap)
a = ap.parse_args()
T.configure(a)
os.makedirs(a.out, exist_ok=True)

m = joblib.load(a.model)
if not a.tabpfn_version:
    T.TABPFN_VERSION = m.get("tabpfn_version", T.TABPFN_VERSION)
if T.TabPFNClassifier is None:
    raise SystemExit("[DỪNG] Script này cần TabPFN thật (GPU); hiện không dùng được TabPFN.")
df = T.engineer(pd.read_csv(a.data, parse_dates=["assigned_date"])).sort_values("assigned_date").reset_index(drop=True)
y = df[T.LABEL].astype(int).values
X = df.drop(columns=[c for c in T.ID_COLS + T.TARGETS if c in df.columns])
t1, t2 = df.assigned_date.quantile(0.60), df.assigned_date.quantile(0.75)
te = (df.assigned_date > t2).values
meta = m.get("meta") or {}
ref = meta.get("test_after") or meta.get("data_end")
if ref and str(t2.date()) != ref:
    print(f"[CẢNH BÁO] Mốc test ({t2.date()}) khác mốc của mô hình ({ref}): file dữ liệu có thể khác lúc huấn luyện.")
Xt = T.restore_dups(X.loc[te].copy(), m["features"]).reindex(columns=m["features"])
print(f"Mô hình: {m.get('model_name')} {m.get('tabpfn_version', '')} | {len(m['contexts'])} túi | ngưỡng {m['thresholds']} | "
      f"test: {int(te.sum()):,} việc (> {t2.date()}), tỉ lệ trễ {y[te].mean():.3f}")
ck = os.path.join(a.out, "checkpoint_report")
os.makedirs(ck, exist_ok=True)
p, sd = T.prob_detail(m, Xt, ckpt=ck, tag="te")
yt = y[te]

res = T.metrics(yt, p)
grp = df.loc[te, "lecturer_id"].values if "lecturer_id" in df.columns else None
ci = T.boot_auc(yt, p, groups=grp)
print(f"\nAUC {res['roc_auc']:.4f} (KTC 95% theo giảng viên {ci[0]:.3f}-{ci[1]:.3f}) | PR-AUC {res['pr_auc']:.3f} | Brier {res['brier']:.4f} | ECE {res['ece']:.4f}")

levels = m["levels"]
lv = pd.DataFrame({"level": [T.level_of(v, levels) for v in p], "y": yt}).groupby("level").y.agg(["count", "mean"]).reindex(T.LEVEL_NAMES)
lv.columns = ["so_viec", "ti_le_tre_thuc_te"]
lv["so_viec"] = lv["so_viec"].fillna(0)
lv["ti_le_viec"] = lv.so_viec / lv.so_viec.sum()
lv["dự_đoán_TB"] = pd.Series(p).groupby([T.level_of(v, levels) for v in p]).mean().reindex(T.LEVEL_NAMES)
print(f"\n=== MỨC CẢNH BÁO (ngưỡng {m['thresholds']}) ===\n" + lv.round(3).to_string())
low, high = lv.loc["Thấp", "ti_le_tre_thuc_te"], lv.loc["Cao", "ti_le_tre_thuc_te"]
ok_low, ok_high = bool(low <= a.max_late_low), bool(high >= a.min_prec_high)
print(f"Mục tiêu: nhóm Thấp trễ <= {a.max_late_low:.0%} -> thực tế {low:.1%} ({'đạt' if ok_low else 'KHÔNG đạt'}); "
      f"nhóm Cao chính xác >= {a.min_prec_high:.0%} -> thực tế {high:.1%} ({'đạt' if ok_high else 'KHÔNG đạt'})")

dec = pd.DataFrame({"p": p, "y": yt})
dec["nhom"] = pd.qcut(dec.p, 10, duplicates="drop")
dtab = dec.groupby("nhom", observed=True).agg(du_doan_TB=("p", "mean"), ti_le_tre_thuc=("y", "mean"), n=("y", "size"))
print("\n=== HIỆU CHUẨN THEO DECILE ===\n" + dtab.round(3).to_string(index=False))
print(f"Dự đoán TB {p.mean():.3f} so với tỉ lệ trễ thực {yt.mean():.3f} (lệch {yt.mean() - p.mean():+.3f})")

seg = {}
seg_defs = [
    ("giảng viên mới (lịch sử < 5 việc)", (Xt.get("hist_count", pd.Series(99, index=Xt.index)) < 5).values),
    ("giảng viên có lịch sử >= 5 việc", (Xt.get("hist_count", pd.Series(0, index=Xt.index)) >= 5).values),
    ("mùa cao điểm", (Xt.get("peak_season", pd.Series(0, index=Xt.index)) == 1).values),
    ("độ phức tạp cao", (Xt.get("complexity", pd.Series("", index=Xt.index)) == "Cao").values),
]
if "checkpoint_fraction" in Xt.columns:
    seg_defs += [
        ("mốc kiểm tra 25% thời gian (checkpoint 0.25)", (Xt["checkpoint_fraction"] == 0.25).values),
        ("mốc kiểm tra 50% thời gian (checkpoint 0.50)", (Xt["checkpoint_fraction"] == 0.50).values),
    ]
if "progress_gap" in Xt.columns:
    seg_defs.append(("chậm tiến độ > 10% (progress_gap < -10)", (Xt["progress_gap"].fillna(0) < -10).values))
if "progress_missing" in Xt.columns:
    seg_defs.append(("chưa cập nhật tiến độ (progress_missing = 1)", (Xt["progress_missing"] == 1).values))
for sname, mask in seg_defs:
    yy, pp = yt[mask], p[mask]
    seg[sname] = dict(n=int(mask.sum()), late_rate=round(float(yy.mean()), 3) if mask.sum() else None,
                      mean_pred=round(float(pp.mean()), 3) if mask.sum() else None,
                      auc=round(float(roc_auc_score(yy, pp)), 3) if mask.sum() >= 50 and 15 <= yy.sum() < len(yy) else None)
print("\n=== HIỆU QUẢ THEO NHÓM ===\n" + pd.DataFrame(seg).T.to_string())

ids = df.loc[te, [c for c in T.ID_COLS if c in df.columns]].reset_index(drop=True)
out = ids.assign(risk_probability=p.round(4), risk_level=[T.level_of(v, levels) for v in p], thuc_te_tre_han=yt)
if len(m["contexts"]) > 1:
    out["risk_uncertainty"] = sd.round(4)
out.to_csv(os.path.join(a.out, "du_doan_tap_test.csv"), index=False, encoding="utf-8-sig")
json.dump(dict(model=a.model, n_bags=len(m["contexts"]), thresholds=m["thresholds"], test=dict((k, round(float(v), 4)) for k, v in res.items()),
               test_auc_ci95_by_lecturer=[round(c, 4) for c in ci],
               alert_levels={k: dict(n=int(r.so_viec), late_rate=T._nn(r.ti_le_tre_thuc_te), share=round(float(r.ti_le_viec), 4)) for k, r in lv.iterrows()},
               target_check=dict(max_late_low=a.max_late_low, actual_low=T._nn(low), ok_low=ok_low, min_prec_high=a.min_prec_high, actual_high=T._nn(high), ok_high=ok_high),
               mean_pred=round(float(p.mean()), 4), late_rate=round(float(yt.mean()), 4), segments=seg,
               calibration_deciles=[dict(mean_pred=round(float(r.du_doan_TB), 4), late_rate=round(float(r.ti_le_tre_thuc), 4), n=int(r.n)) for r in dtab.itertuples()]),
          open(os.path.join(a.out, "bao_cao_muc_canh_bao.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
if not a.keep_checkpoint:
    shutil.rmtree(ck, ignore_errors=True)
print(f"\nĐã lưu bao_cao_muc_canh_bao.json và du_doan_tap_test.csv vào {os.path.abspath(a.out)}")


In [ ]:
%%writefile evaluate_model.py
#!/usr/bin/env python3
"""
ĐÁNH GIÁ MÔ HÌNH CHÍNH THEO GIẢNG VIÊN MỚI / LỊCH SỬ ÍT (cold-start) — chạy sau train_model.py.

  python evaluate_model.py --data ./data/ml_training_set_clean.csv --model ./model/risk_model.joblib --out ./eval

Gồm các phép đánh giá (TabPFN: mỗi fold dựng ngữ cảnh mới, chi phí nằm ở bước dự đoán -> nên dùng GPU):
  A. Tập test THEO THỜI GIAN (giống train_model.py) — báo cáo tách theo số việc đã hoàn thành trước đó (hist_count).
  B. Hold-out THEO GIẢNG VIÊN (GroupKFold 5 phần theo lecturer_id): mô hình chưa từng thấy giảng viên trong tập test.
     Dự đoán out-of-fold được gộp lại rồi tách nhóm lịch sử — đủ mẫu cho nhóm giảng viên mới.
  C. Nhóm giảng viên mới theo thâm niên tại Khoa (tenure_days < 90) trong phép B.
  D. Hold-out THEO KHOA (leave-one-faculty-out): mô hình chưa từng thấy Khoa nào trong tập test (chuyển miền).
Mỗi nhóm báo: n, tỉ lệ trễ thực tế, xác suất dự đoán trung bình (mean_pred), AUC (+ khoảng tin cậy bootstrap), Brier và ECE của MÔ HÌNH CHÍNH (TabPFN; không có mô hình nào khác).
Nhóm quá ít mẫu (n < 100 hoặc < 15 việc trễ) chỉ báo n và tỉ lệ trễ.
"""
import argparse, json, os, warnings
import numpy as np, pandas as pd, joblib
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.model_selection import GroupKFold
import train_model as T
warnings.filterwarnings("ignore")

BUCKETS = [("hist_count = 0", lambda h: h == 0), ("hist_count 1-4", lambda h: (h >= 1) & (h < 5)),
           ("hist_count 5-19", lambda h: (h >= 5) & (h < 20)), ("hist_count >= 20", lambda h: h >= 20)]


def seg(y, p):
    n, pos = len(y), int(y.sum()); r = dict(n=n, late_rate=round(float(y.mean()), 3) if n else None, mean_pred=round(float(p.mean()), 3) if n else None, auc=None, auc_ci95=None, brier=None, ece=None)
    if n: r["brier"] = round(float(brier_score_loss(y, p)), 3)
    if n >= 100 and pos >= 15 and n - pos >= 15:
        r["auc"] = round(float(roc_auc_score(y, p)), 3); r["auc_ci95"] = [round(c, 3) for c in T.boot_auc(y, p, n=200)]
        r["ece"] = round(T.ece(y, p, k=10 if n >= 1000 else 5), 3)
    return r


def table(y, p, h, tenure):
    rows = {"TẤT CẢ": seg(y, p)}
    for name, f in BUCKETS:
        m = f(h); rows[name] = seg(y[m], p[m])
    if tenure is not None:
        m = tenure < 90; rows["thâm niên tại Khoa < 90 ngày"] = seg(y[m], p[m])
    return pd.DataFrame(rows).T


def main():
    if T.TabPFNClassifier is None:
        raise SystemExit("[DỪNG] evaluate_model.py cần TabPFN thật (GPU).")
    ap = argparse.ArgumentParser(); ap.add_argument("--data", required=True); ap.add_argument("--model", default=None)
    ap.add_argument("--out", default="./eval"); ap.add_argument("--folds", type=int, default=5); T.add_tabpfn_args(ap); a = ap.parse_args(); T.configure(a)
    df = T.engineer(pd.read_csv(a.data, parse_dates=["assigned_date"])).sort_values("assigned_date").reset_index(drop=True)
    y = df[T.LABEL].astype(int).values; X = df.drop(columns=[c for c in T.ID_COLS + T.TARGETS if c in df.columns])
    h = df["hist_count"].values; ten = df["tenure_days"].values if "tenure_days" in df else None
    if a.model: mj = joblib.load(a.model); cols = mj["features"]; X = T.restore_dups(X, cols); T.TABPFN_VERSION = a.tabpfn_version or mj.get("tabpfn_version", T.TABPFN_VERSION)
    else: cols = list(X.columns); mj = None
    os.makedirs(a.out, exist_ok=True); out = {}

    # A. Tập test theo thời gian: dùng đúng mô hình đã lưu (huấn luyện trên train+validation)
    t2 = df.assigned_date.quantile(0.75); te = (df.assigned_date > t2).values
    if a.model: pA = T.prob(mj, X.loc[te, cols])
    else: pA = T.prob(T.fit_model(X.loc[~te, cols], y[~te]), X.loc[te, cols])
    tA = table(y[te], pA, h[te], ten[te] if ten is not None else None)
    print(f"=== A. TẬP TEST THEO THỜI GIAN (n = {te.sum()}) ===\n" + tA.drop(columns="auc_ci95").to_string()); out["A_temporal_test"] = tA.to_dict("index")

    # B/C. Hold-out theo giảng viên
    oof = np.zeros(len(df))
    for k, (tr_i, te_i) in enumerate(GroupKFold(a.folds).split(X, y, df.lecturer_id.values)):
        m = T.fit_model(X.iloc[tr_i][cols], y[tr_i]); oof[te_i] = T.prob(m, X.iloc[te_i][cols])
    tB = table(y, oof, h, ten)
    print(f"\n=== B+C. HOLD-OUT THEO GIẢNG VIÊN ({a.folds}-fold, mô hình chưa từng thấy giảng viên trong tập test; n = {len(df)}) ===\n" + tB.drop(columns="auc_ci95").to_string())
    out["B_unseen_lecturers"] = tB.to_dict("index")
    print("\nKhoảng tin cậy 95% bootstrap của AUC (hold-out theo giảng viên; bootstrap theo dòng nên hơi lạc quan):")
    for k, r in tB.iterrows():
        if r["auc_ci95"] is not None: print(f"   {k:34s} {r['auc']:.3f}  {r['auc_ci95']}")
    # chênh lệch giữa nhóm giảng viên mới và nhóm có lịch sử
    new = tB.loc["hist_count 1-4", "auc"], tB.loc["hist_count >= 20", "auc"]
    if None not in new: print(f"\nChênh lệch AUC giữa nhóm lịch sử 1-4 việc và >= 20 việc: {new[0] - new[1]:+.3f}")
    if "faculty" in df.columns and df.faculty.nunique() >= 3:
        oofd = np.zeros(len(df)); fac = df.faculty.values; rows = []
        for f in sorted(df.faculty.unique()):
            te_f = fac == f
            m = T.fit_model(X.loc[~te_f, cols], y[~te_f]); oofd[te_f] = T.prob(m, X.loc[te_f, cols])
            rows.append(dict(khoa=f, n=int(te_f.sum()), late_rate=round(float(y[te_f].mean()), 3), mean_pred=round(float(oofd[te_f].mean()), 3),
                             auc=round(float(roc_auc_score(y[te_f], oofd[te_f])), 3), brier=round(float(brier_score_loss(y[te_f], oofd[te_f])), 3)))
        tD = pd.DataFrame(rows).set_index("khoa")
        print(f"\n=== D. CHUYỂN SANG KHOA CHƯA TỪNG THẤY (leave-one-faculty-out, {df.faculty.nunique()} Khoa) ===\n" + tD.to_string())
        print(f"AUC gộp: {roc_auc_score(y, oofd):.3f} | AUC theo Khoa: min {tD.auc.min():.3f}, max {tD.auc.max():.3f} | Brier gộp: {brier_score_loss(y, oofd):.3f}")
        out["D_unseen_faculty"] = tD.to_dict("index")
    json.dump(out, open(f"{a.out}/evaluation.json", "w"), ensure_ascii=False, indent=2, default=str)
    pd.concat({"A_test_theo_thoi_gian": tA, "B_hold_out_theo_giang_vien": tB}).to_csv(f"{a.out}/evaluation_table.csv", encoding="utf-8-sig")
    print(f"\nĐã lưu vào {os.path.abspath(a.out)}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile clean_data.py
#!/usr/bin/env python3
"""
Làm sạch + kiểm định tập dữ liệu huấn luyện/dự đoán rủi ro trễ hạn (chạy MỘT lần trước khi train).

  python clean_data.py --input ./data/ml_training_set.csv --output ./data/ml_training_set_clean.csv
  python clean_data.py --input ./data/ml_dynamic_set.csv  --output ./data/ml_dynamic_set_clean.csv
  python clean_data.py --input viec_moi.csv --output viec_moi_clean.csv --inference     # dữ liệu cần dự đoán (không có nhãn)

Nguyên tắc: KHÔNG điền giá trị thiếu và KHÔNG xóa giá trị cực đoan hợp lệ (đó là tín hiệu cho mô hình);
chỉ loại bản ghi sai/trùng, chuẩn hóa văn bản, giới hạn đuôi dài, bỏ cột trùng thông tin và kiểm tra rò rỉ.
Dữ liệu cần dự đoán PHẢI đi qua cùng script này (với --inference) để có cùng các cột như lúc huấn luyện.
"""
import argparse, json, os, unicodedata
import numpy as np, pandas as pd

CFG = dict(
    id_cols=["task_id", "lecturer_id", "faculty", "assigned_date"],
    targets=["is_late", "late_vs_original_deadline", "late_vs_current_deadline"],
    core_required=["task_type", "complexity", "priority", "available_days", "expected_duration_days",
                   "deadline_pressure", "hist_count", "hist_late_rate", "active_tasks",
                   "workload_next_14d", "deadlines_next_7d"],
    caps={"deadline_pressure": 5.0, "workload_next_14d": 500.0, "active_tasks": 20, "active_high_priority_tasks": 15},
    log_cols=["available_days", "expected_duration_days"],            # thêm log1p, giữ nguyên cột gốc
    drop_pairs=[("working_days_available", "available_days"),         # (cột bỏ, cột giữ) nếu |r| > corr_thr
                ("estimated_effort_hours", "expected_duration_days"),
                ("course_count", "teaching_hours_semester")],
    corr_thr=0.90,
    missing_flag_cols=["converted_workload", "hist_avg_delay", "hist_early_rate", "hist_late_last5"],
    nonneg_cols=["available_days", "expected_duration_days", "hist_count", "active_tasks", "workload_next_14d",
                 "deadlines_next_7d", "deadlines_next_14d", "subtask_count", "dependency_count", "team_size", "tenure_days"],
    late_rate_range=(0.10, 0.35),
)


def norm_text(s):
    s2 = s.astype("object")
    f = lambda x: x if (x is None or (isinstance(x, float) and np.isnan(x)) or x is pd.NA) else \
        " ".join(unicodedata.normalize("NFC", str(x)).split())
    return s2.map(f)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", required=True)
    ap.add_argument("--output", default=None)
    ap.add_argument("--inference", action="store_true", help="dữ liệu không có cột nhãn")
    ap.add_argument("--add-log", action="store_true", help="thêm log1p của available_days, expected_duration_days (mặc định tắt)")
    a = ap.parse_args()
    out = a.output or os.path.splitext(a.input)[0] + "_clean.csv"
    rep = dict(input=a.input, output=out, steps={}, warnings=[])
    warn = lambda m: (rep["warnings"].append(m), print("  [CẢNH BÁO]", m))

    df = pd.read_csv(a.input)
    n0 = len(df); rep["rows_in"] = n0
    print(f"Đọc {n0} dòng, {df.shape[1]} cột từ {a.input}")

    # 1. Kiểm tra schema
    need = CFG["core_required"] + ([] if a.inference else ["is_late"])
    miss = [c for c in need if c not in df.columns]
    if miss:
        raise SystemExit(f"LỖI: thiếu cột bắt buộc: {miss}")
    if "assigned_date" in df.columns:
        df["assigned_date"] = pd.to_datetime(df["assigned_date"], errors="coerce")
    print("1. Schema: OK")

    # 2. Chuẩn hóa văn bản (NFC, bỏ khoảng trắng thừa)
    txt = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c]) and c not in ("assigned_date",)]
    for c in txt: df[c] = norm_text(df[c])
    print(f"2. Chuẩn hóa văn bản: {len(txt)} cột")

    # 3. Loại bản ghi trùng
    key = [c for c in ["task_id", "lecturer_id", "checkpoint_fraction"] if c in df.columns]
    d_full = int(df.duplicated().sum()); df = df.drop_duplicates()
    d_key = int(df.duplicated(key).sum()) if key else 0
    if key: df = df.drop_duplicates(key)
    rep["steps"]["duplicates_removed"] = dict(full_row=d_full, by_key=d_key, key=key)
    print(f"3. Trùng lặp: bỏ {d_full} dòng trùng hoàn toàn, {d_key} dòng trùng khóa {key}")

    # 4. Loại bản ghi sai nghiệp vụ
    bad = pd.Series(False, index=df.index); reasons = {}
    def flag(mask, name):
        nonlocal bad
        mask = mask.fillna(False); reasons[name] = int(mask.sum()); bad |= mask
    if "assigned_date" in df: flag(df.assigned_date.isna(), "assigned_date_khong_hop_le")
    flag(df.available_days < 1, "available_days_<1")
    flag(~df.hist_late_rate.between(0, 1) & df.hist_late_rate.notna(), "hist_late_rate_ngoai_[0,1]")
    for c in CFG["nonneg_cols"]:
        if c in df: flag(df[c] < 0, f"{c}_am")
    if not a.inference:
        for t in CFG["targets"]:
            if t in df: flag(~df[t].isin([0, 1]) & df[t].notna(), f"{t}_khong_phai_0/1")
        flag(df.is_late.isna(), "thieu_nhan")
    df = df[~bad].copy()
    rep["steps"]["invalid_rows_removed"] = dict(total=int(bad.sum()), reasons={k: v for k, v in reasons.items() if v})
    print(f"4. Sai nghiệp vụ: bỏ {int(bad.sum())} dòng {({k: v for k, v in reasons.items() if v}) or ''}")

    # 5. Giá trị thiếu: chỉ báo cáo + thêm cờ cho nhóm thiếu có ý nghĩa nghiệp vụ (KHÔNG điền)
    na = (df.isna().mean() * 100).round(2); na = na[na > 0]
    flags = []
    for c in CFG["missing_flag_cols"]:
        if c in df: df[f"{c}_missing"] = df[c].isna().astype(int); flags.append(f"{c}_missing")
    df["is_cold_start"] = (df.hist_count == 0).astype(int)
    rep["steps"]["missing_pct"] = na.to_dict(); rep["steps"]["flags_added"] = flags + ["is_cold_start"]
    print(f"5. Giá trị thiếu ở {len(na)} cột (giữ nguyên, thêm cờ: {flags + ['is_cold_start']})")

    # 6. Giới hạn đuôi dài + thêm cột log
    clipped = {}
    for c, cap in CFG["caps"].items():
        if c in df:
            n = int((df[c] > cap).sum()); clipped[c] = n
            df[c] = df[c].clip(upper=cap)
    if a.add_log:
        for c in CFG["log_cols"]:
            if c in df: df[f"log_{c}"] = np.log1p(df[c])
    rep["steps"]["clipped"] = clipped
    print(f"6. Giới hạn đuôi dài: {clipped} | cột log: {'bật' if a.add_log else 'tắt'}")

    # 7. Bỏ cột trùng thông tin
    dropped = []
    for drop, keep in CFG["drop_pairs"]:
        if drop in df and keep in df:
            r = df[drop].corr(df[keep])
            if abs(r) > CFG["corr_thr"]:
                df = df.drop(columns=drop); dropped.append(dict(drop=drop, keep=keep, corr=round(float(r), 3)))
    rep["steps"]["dropped_redundant"] = dropped
    print(f"7. Bỏ cột trùng thông tin: {[d['drop'] for d in dropped]}")

    # 8. Kiểm tra rò rỉ & tính nhất quán thời gian
    if {"lecturer_id", "assigned_date", "hist_count"} <= set(df.columns):
        s = df.sort_values(["lecturer_id", "assigned_date"])
        viol = int((s.groupby("lecturer_id").hist_count.diff() < 0).sum())
        first = s.groupby("lecturer_id").head(1)
        if viol: warn(f"hist_count giảm theo thời gian ở {viol} dòng -> nghi ngờ tính lịch sử sai")
        if "tenure_days" in df and ((df.hist_count > 0) & (df.tenure_days <= 0)).any():
            warn("Có dòng thâm niên 0 ngày nhưng đã có lịch sử công việc -> kiểm tra cách tính đặc trưng lịch sử")
        rep["steps"]["temporal_violations"] = viol
    banned = [c for c in ["status", "progress_pct_final", "actual_completion_date", "completed_date",
                          "delay_days", "delay_days_vs_original", "extension_requested", "extension_days"] if c in df.columns]
    if banned: warn(f"Cột có thể gây rò rỉ nhãn: {banned}")
    cold = float(df.is_cold_start.mean()); low = float((df.hist_count < 5).mean())
    n_low = int((df.hist_count < 5).sum())
    if n_low < 500: warn(f"Chỉ {n_low} dòng có lịch sử < 5 việc ({low:.1%}) -> chưa đủ để đánh giá giảng viên mới")
    rep["steps"]["cold_start"] = dict(hist0=round(cold, 4), hist_lt5=round(low, 4))
    print(f"8. Rò rỉ/thời gian: {len(rep['warnings'])} cảnh báo | cold-start {cold:.1%} | lịch sử<5: {low:.1%}")

    # 9. Nhãn
    if not a.inference:
        r = float(df.is_late.mean()); lo, hi = CFG["late_rate_range"]
        rep["late_rate"] = round(r, 4)
        if not lo <= r <= hi: warn(f"Tỉ lệ trễ {r:.1%} ngoài khoảng {lo:.0%}-{hi:.0%}")
        print(f"9. Nhãn: tỉ lệ trễ {r:.1%}")

    df.to_csv(out, index=False, encoding="utf-8-sig")
    rep["rows_out"] = len(df); rep["cols_out"] = df.shape[1]
    jp = os.path.splitext(out)[0] + "_report.json"
    json.dump(rep, open(jp, "w", encoding="utf-8"), ensure_ascii=False, indent=2, default=str)
    print(f"\nKết quả: {n0} -> {len(df)} dòng, {df.shape[1]} cột. Đã ghi {out} và {jp}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile build_task_personnel_matrix.py
#!/usr/bin/env python3
"""
MA TRẬN TÁC VỤ – NHÂN SỰ (task–personnel feature matrix)

  python build_task_personnel_matrix.py --data ./data/ml_training_set_clean.csv --out ./data
  python build_task_personnel_matrix.py --data ./data/ml_training_set_clean.csv --out ./data --enrich ./data/ml_training_set_clean.csv

Ý nghĩa (đúng đề cương khóa luận):
  Mỗi ô (giảng viên × loại công việc) tóm tắt hiệu suất lịch sử:
    - số lần đã làm loại việc đó
    - tỉ lệ trễ
    - số ngày trễ trung bình (khi trễ)
  Dùng để:
    1) Phân tích / heat-map (báo cáo)
    2) Bổ sung đặc trưng cho mô hình dự đoán (lecturer_task_late_rate, ...)
    3) Prior khi cold-start: chưa từng làm loại việc → lấy trung bình Bộ môn cùng loại việc

Tránh rò rỉ thời gian: đặc trưng của dòng t chỉ dùng các việc hoàn thành TRƯỚC assigned_date của dòng t.
"""
import argparse, json, os
import numpy as np
import pandas as pd

PRIOR_RATE = 0.20
PRIOR_STRENGTH = 3.0   # Laplace smoothing
MIN_COUNT_PERSONAL = 1  # dưới ngưỡng này ưu tiên prior Bộ môn


def _smooth_rate(late_sum, count, prior=PRIOR_RATE, strength=PRIOR_STRENGTH):
    return (late_sum + prior * strength) / (count + strength)


def build_static_matrix(df, key_task="task_type"):
    """Ma trận tổng hợp toàn tập (chỉ để báo cáo / heat-map, KHÔNG dùng trực tiếp làm đặc trưng huấn luyện)."""
    g = df.groupby(["lecturer_id", key_task], dropna=False)
    mat = g.agg(
        n_tasks=("is_late", "size"),
        n_late=("is_late", "sum"),
        late_rate=("is_late", "mean"),
    ).reset_index()
    mat["late_rate_smooth"] = _smooth_rate(mat.n_late, mat.n_tasks)
    if "department" in df.columns:
        lect_dep = df[["lecturer_id", "department"]].drop_duplicates("lecturer_id")
        mat = mat.merge(lect_dep, on="lecturer_id", how="left")
        dep = df.groupby(["department", key_task], dropna=False).agg(
            dept_n=("is_late", "size"), dept_late=("is_late", "sum")
        ).reset_index()
        dep["dept_late_rate_smooth"] = _smooth_rate(dep.dept_late, dep.dept_n)
        mat = mat.merge(dep, on=["department", key_task], how="left")
    mat["key_task"] = key_task
    return mat.sort_values(["lecturer_id", key_task]).reset_index(drop=True)


def enrich_temporal(df, key_task="task_type", out_dir=None, data_path=None):
    """
    Gắn đặc trưng ma trận vào từng dòng, chỉ dùng lịch sử của các việc ĐÃ HOÀN THÀNH TRƯỚC ngày giao (tránh rò rỉ).
    Nếu bảng dữ liệu chưa có completed_date / expected_duration_days, tự động tìm trong task_assignments.csv cùng thư mục,
    hoặc xấp xỉ ngày hoàn thành tối thiểu = assigned_date + available_days.
    Trả về df kèm các cột:
      lecturer_task_count, lecturer_task_late_rate,
      dept_task_late_rate, task_matrix_source  (personal | department | global)
    """
    df = df.sort_values(["assigned_date", "lecturer_id"]).reset_index(drop=True)
    n = len(df)
    out_count = np.zeros(n, dtype=float)
    out_rate = np.full(n, PRIOR_RATE)
    out_dept = np.full(n, PRIOR_RATE)
    out_src = np.array(["global"] * n, dtype=object)

    # Xác định thời điểm hoàn thành (done_date) của từng dòng để chỉ cập nhật lịch sử sau khi việc đã xong
    done_dates = None
    if "completed_date" in df.columns:
        done_dates = pd.to_datetime(df["completed_date"], errors="coerce")
    elif data_path:
        ta_path = os.path.join(os.path.dirname(os.path.abspath(data_path)), "task_assignments.csv")
        if os.path.exists(ta_path) and {"task_id", "lecturer_id"} <= set(df.columns):
            try:
                ta = pd.read_csv(ta_path, usecols=["task_id", "lecturer_id", "completed_date"], parse_dates=["completed_date"])
                ta = ta.drop_duplicates(["task_id", "lecturer_id"])
                merged_cd = df[["task_id", "lecturer_id"]].merge(ta, on=["task_id", "lecturer_id"], how="left")["completed_date"]
                if merged_cd.notna().any():
                    done_dates = merged_cd
            except Exception:
                done_dates = None
    if done_dates is None and "assigned_date" in df.columns:
        ad = pd.to_datetime(df["assigned_date"], errors="coerce")
        dur = df["available_days"].fillna(1) if "available_days" in df.columns else 0
        done_dates = ad + pd.to_timedelta(dur, unit="D")

    # state: (lecturer_id, task) -> [count, late_sum]
    #        (department, task) -> [count, late_sum]
    lect_state = {}
    dept_state = {}
    global_state = {}  # task -> [count, late_sum]

    has_dept = "department" in df.columns
    import heapq
    pending = []  # min-heap theo (done_ord, idx, lk, dk, task, late)
    ad_ords = pd.to_datetime(df["assigned_date"]).map(lambda x: x.toordinal() if pd.notna(x) else 0).to_numpy() if "assigned_date" in df.columns else np.arange(n)
    done_ords = done_dates.map(lambda x: x.toordinal() if pd.notna(x) else 0).to_numpy() if done_dates is not None else ad_ords

    for i, r in enumerate(df.itertuples(index=False)):
        cur_ord = ad_ords[i]
        # Đưa các việc đã hoàn thành TRƯỚC ngày giao hiện tại (done_ord < cur_ord) vào lịch sử
        while pending and pending[0][0] < cur_ord:
            _, _, p_lk, p_dk, p_task, p_late = heapq.heappop(pending)
            if p_lk not in lect_state:
                lect_state[p_lk] = [0.0, 0.0]
            lect_state[p_lk][0] += 1
            lect_state[p_lk][1] += p_late
            if p_dk is not None:
                if p_dk not in dept_state:
                    dept_state[p_dk] = [0.0, 0.0]
                dept_state[p_dk][0] += 1
                dept_state[p_dk][1] += p_late
            if p_task not in global_state:
                global_state[p_task] = [0.0, 0.0]
            global_state[p_task][0] += 1
            global_state[p_task][1] += p_late

        lid = r.lecturer_id
        task = getattr(r, key_task)
        dep = getattr(r, "department", None) if has_dept else None

        lk = (lid, task)
        if lk in lect_state:
            c, s = lect_state[lk]
            out_count[i] = c
            out_rate[i] = _smooth_rate(s, c)
            out_src[i] = "personal"
        else:
            out_count[i] = 0
            # fallback department
            if has_dept and dep is not None and (dep, task) in dept_state:
                c, s = dept_state[(dep, task)]
                out_rate[i] = _smooth_rate(s, c)
                out_src[i] = "department"
            elif task in global_state:
                c, s = global_state[task]
                out_rate[i] = _smooth_rate(s, c)
                out_src[i] = "global"
            else:
                out_rate[i] = PRIOR_RATE
                out_src[i] = "global"

        if has_dept and dep is not None and (dep, task) in dept_state:
            c, s = dept_state[(dep, task)]
            out_dept[i] = _smooth_rate(s, c)
        elif task in global_state:
            c, s = global_state[task]
            out_dept[i] = _smooth_rate(s, c)
        else:
            out_dept[i] = PRIOR_RATE

        # Đưa kết quả của dòng hiện tại vào hàng đợi chờ tới ngày hoàn thành (done_ord)
        late = float(r.is_late) if hasattr(r, "is_late") and r.is_late == r.is_late else 0.0
        dk = (dep, task) if (has_dept and dep is not None) else None
        heapq.heappush(pending, (done_ords[i], i, lk, dk, task, late))

    df = df.copy()
    df["lecturer_task_count"] = out_count
    df["lecturer_task_late_rate"] = out_rate
    df["dept_task_late_rate"] = out_dept
    df["task_matrix_source"] = out_src
    # chênh lệch so với prior toàn cục: dương = rủi ro cao hơn mặt bằng loại việc
    df["lecturer_task_late_lift"] = df["lecturer_task_late_rate"] - df["dept_task_late_rate"]
    return df


def pivot_heatmap(mat, key_task="task_type", value="late_rate_smooth", top_lecturers=15, top_tasks=12):
    """Ma trận rộng để vẽ heat-map / đưa vào báo cáo."""
    # chọn GV và task phổ biến nhất
    top_l = mat.groupby("lecturer_id").n_tasks.sum().nlargest(top_lecturers).index
    top_t = mat.groupby(key_task).n_tasks.sum().nlargest(top_tasks).index
    sub = mat[mat.lecturer_id.isin(top_l) & mat[key_task].isin(top_t)]
    wide = sub.pivot_table(index="lecturer_id", columns=key_task, values=value, aggfunc="first")
    return wide


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="ml_training_set_clean.csv (có is_late, task_type, lecturer_id)")
    ap.add_argument("--out", default="./data")
    ap.add_argument("--key-task", default="task_type", choices=["task_type", "task_group"])
    ap.add_argument("--enrich", default=None,
                    help="file CSV cần gắn đặc trưng ma trận (mặc định = --data); ghi *_with_matrix.csv")
    ap.add_argument("--inference", action="store_true",
                    help="file enrich không có nhãn: chỉ merge ma trận tĩnh đã tính từ --data")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)

    df = pd.read_csv(a.data, parse_dates=["assigned_date"] if "assigned_date" in pd.read_csv(a.data, nrows=0).columns else None)
    is_prebuilt_mat = {"lecturer_id", a.key_task, "n_tasks", "late_rate_smooth"} <= set(df.columns)
    if "is_late" not in df.columns and not is_prebuilt_mat and not a.inference:
        raise SystemExit("Cần cột is_late (hoặc file ma trận đã tính sẵn task_personnel_matrix.csv) để xây ma trận. Với dữ liệu dự đoán: truyền --data là tập có nhãn hoặc task_personnel_matrix.csv + --enrich file cần dự đoán --inference")

    # 1) Ma trận tĩnh (báo cáo & tra cứu khi inference)
    if "is_late" in df.columns:
        df_mat_src = df
        if "assigned_date" in df.columns:
            # Chỉ tổng hợp trên phần Train+Validation (<= phân vị 75%) để tránh rò rỉ nhãn tập Test vào ma trận tĩnh khi inference
            t2 = df["assigned_date"].quantile(0.75)
            df_mat_src = df[df["assigned_date"] <= t2].copy()
        mat = build_static_matrix(df_mat_src, key_task=a.key_task)
        mat_path = os.path.join(a.out, "task_personnel_matrix.csv")
        mat.to_csv(mat_path, index=False, encoding="utf-8-sig")
        print(f"Ma trận tĩnh: {len(mat)} ô (lecturer × {a.key_task}, từ {len(df_mat_src)} dòng lịch sử) → {mat_path}")
        print(mat.head(8).to_string(index=False))

        wide = pivot_heatmap(mat, key_task=a.key_task)
        wide_path = os.path.join(a.out, "task_personnel_matrix_wide.csv")
        wide.to_csv(wide_path, encoding="utf-8-sig")
        print(f"Ma trận wide (heat-map): {wide.shape} → {wide_path}")

        # thống kê nhanh
        coverage = (mat.n_tasks >= 3).mean()
        print(f"Tỉ lệ ô có ≥ 3 việc: {coverage:.1%} | late_rate_smooth TB: {mat.late_rate_smooth.mean():.3f}")
    elif is_prebuilt_mat:
        mat = df
        print(f"Nạp trực tiếp ma trận tĩnh đã tính sẵn: {len(mat)} ô từ {a.data}")
    else:
        mat_fallback = os.path.join(a.out, "task_personnel_matrix.csv")
        if os.path.exists(mat_fallback):
            mat = pd.read_csv(mat_fallback)
            print(f"Nạp ma trận tĩnh từ {mat_fallback}: {len(mat)} ô")
        else:
            mat = None
            print("Bỏ qua ma trận tĩnh (không có nhãn trong --data)")

    # 2) Enrich đặc trưng theo thời gian
    enrich_path = a.enrich or a.data
    if is_prebuilt_mat and not a.enrich:
        print("Đã nạp ma trận tĩnh (không có --enrich cần gắn đặc trưng). Xong.")
        return
    edf = pd.read_csv(enrich_path, parse_dates=["assigned_date"] if "assigned_date" in pd.read_csv(enrich_path, nrows=0).columns else None)

    if a.inference or "is_late" not in edf.columns:
        # merge ma trận tĩnh (đã tính từ tập có nhãn)
        if mat is None:
            raise SystemExit("Cần --data có nhãn hoặc file task_personnel_matrix.csv để suy ra ma trận khi --inference")
        # tránh trùng tên cột nếu file enrich đã có sẵn các cột ma trận
        drop_existing = [c for c in ["lecturer_task_count", "lecturer_task_late_rate", "dept_task_late_rate",
                                     "task_matrix_source", "lecturer_task_late_lift"] if c in edf.columns]
        if drop_existing:
            edf = edf.drop(columns=drop_existing)
        key = ["lecturer_id", a.key_task]
        use = mat[key + ["n_tasks", "late_rate_smooth"]].drop_duplicates(key).rename(
            columns={"n_tasks": "lecturer_task_count", "late_rate_smooth": "lecturer_task_late_rate"}
        )
        edf = edf.merge(use, on=key, how="left")
        if "department" in edf.columns and "dept_late_rate_smooth" in mat.columns:
            dep = mat[["department", a.key_task, "dept_late_rate_smooth"]].drop_duplicates(["department", a.key_task])
            dep = dep.rename(columns={"dept_late_rate_smooth": "dept_task_late_rate"})
            edf = edf.merge(dep, on=["department", a.key_task], how="left")
        else:
            edf["dept_task_late_rate"] = PRIOR_RATE
        has_personal = edf["lecturer_task_count"].fillna(0) > 0
        has_dept_prior = edf["dept_task_late_rate"].notna()
        edf["lecturer_task_count"] = edf["lecturer_task_count"].fillna(0)
        edf["dept_task_late_rate"] = edf["dept_task_late_rate"].fillna(PRIOR_RATE)
        # Khi giảng viên chưa từng làm loại việc này (count == 0), lùi về tỉ lệ trễ của Bộ môn (nếu có), sau đó mới tới PRIOR_RATE
        edf["lecturer_task_late_rate"] = np.where(has_personal, edf["lecturer_task_late_rate"], edf["dept_task_late_rate"])
        edf["task_matrix_source"] = np.where(has_personal, "personal", np.where(has_dept_prior, "department", "global"))
        edf["lecturer_task_late_lift"] = edf["lecturer_task_late_rate"] - edf["dept_task_late_rate"]
        print(f"[inference] merge ma trận tĩnh vào {len(edf)} dòng")
    else:
        edf = enrich_temporal(edf, key_task=a.key_task, out_dir=a.out, data_path=a.data)
        src_dist = edf.task_matrix_source.value_counts(normalize=True).round(3).to_dict()
        print(f"Đặc trưng thời gian: nguồn prior = {src_dist}")
        print(f"  lecturer_task_count=0 (chưa làm loại việc): {(edf.lecturer_task_count == 0).mean():.1%}")

    out_enrich = os.path.join(a.out, os.path.splitext(os.path.basename(enrich_path))[0] + "_with_matrix.csv")
    # nếu enrich trùng data mặc định
    if os.path.abspath(enrich_path) == os.path.abspath(a.data) and not a.enrich:
        out_enrich = os.path.join(a.out, "ml_training_set_with_matrix.csv")
    edf.to_csv(out_enrich, index=False, encoding="utf-8-sig")
    print(f"Đã ghi file kèm đặc trưng ma trận: {out_enrich}")

    meta = dict(
        key_task=a.key_task,
        prior_rate=PRIOR_RATE,
        prior_strength=PRIOR_STRENGTH,
        n_matrix_cells=int(len(mat)) if mat is not None else 0,
        features_added=[
            "lecturer_task_count", "lecturer_task_late_rate",
            "dept_task_late_rate", "lecturer_task_late_lift", "task_matrix_source",
        ],
        output_matrix=os.path.join(a.out, "task_personnel_matrix.csv"),
        output_enriched=out_enrich,
    )
    json.dump(meta, open(os.path.join(a.out, "task_personnel_matrix_meta.json"), "w", encoding="utf-8"),
              ensure_ascii=False, indent=2)
    print("Xong.")


if __name__ == "__main__":
    main()


## 4. Dữ liệu cố định (giải nén, kiểm tra mã băm)

In [ ]:
import subprocess, pandas as pd
assert os.path.exists('/content/du_lieu_co_dinh_v2_seed44.zip'), 'Upload du_lieu_co_dinh_v2_seed44.zip vào /content hoặc gốc Google Drive'
sh('unzip -q -o /content/du_lieu_co_dinh_v2_seed44.zip -d /content')
r = subprocess.run(['python', 'verify_data.py'], cwd='/content', capture_output=True, text=True)
print(r.stdout[-300:]); assert r.returncode == 0, 'Dữ liệu bị hỏng hoặc khác bản gốc: tải lại du_lieu_co_dinh_v2_seed44.zip'
STATIC_CSV, DYN_CSV = './data/ml_training_set_with_matrix.csv', './data/ml_dynamic_set_clean_with_matrix.csv'
if QUICK:
    for src, dst in [(STATIC_CSV, './data/quick_static.csv'), (DYN_CSV, './data/quick_dynamic.csv')]:
        pd.read_csv(src).iloc[::8].to_csv(dst, index=False)
    STATIC_CSV, DYN_CSV = './data/quick_static.csv', './data/quick_dynamic.csv'
for n, p in [('tĩnh', STATIC_CSV), ('động', DYN_CSV)]:
    d = pd.read_csv(p, usecols=['assigned_date', 'is_late']); print(f'{n}: {len(d):,} dòng | tỉ lệ trễ {d.is_late.mean():.3f}')

## 5. Huấn luyện mô hình TĨNH bằng TabPFN
Nếu TabPFN không dùng được, script sẽ DỪNG (không có chế độ thay thế bằng mô hình khác).

In [ ]:
%%time
if RUN_STATIC:
    sh(f'python train_model.py train --data {STATIC_CSV} --out {OUT}/model {COMMON} {TRAIN_EXTRA}')
else:
    print('RUN_STATIC = False -> bỏ qua')

## 6. Huấn luyện mô hình ĐỘNG bằng TabPFN

In [ ]:
%%time
if RUN_DYNAMIC:
    sh(f'python train_model.py train --data {DYN_CSV} --out {OUT}/model_dynamic {COMMON} {TRAIN_EXTRA}')
else:
    print('RUN_DYNAMIC = False -> bỏ qua')

### Kiểm tra nhãn mô hình (phải là TabPFN thật)

In [ ]:
import json
for n in ['model', 'model_dynamic']:
    p = f'{OUT}/{n}/metrics.json'
    if not os.path.exists(p): print(n, ': chưa có metrics.json'); continue
    m = json.load(open(p, encoding='utf-8')); v = m['meta']['versions']
    ok = m['main_model'] == 'tabpfn' and m['device'] == 'cuda' and v.get('tabpfn') not in (None, '?')
    print(f"{n}: main_model={m['main_model']} | tabpfn={v.get('tabpfn')} | device={m['device']} | pool={m['pool']} | n_bags={m['meta']['settings']['n_bags']} | AUC test {m['test']['roc_auc']:.4f} | thresholds {m['thresholds']} -> {'ĐÚNG TabPFN trên GPU' if ok else ('(chế độ quick/CPU)' if QUICK else 'CẢNH BÁO: KHÔNG PHẢI TabPFN trên GPU')}")

## 7. Mô hình đối chứng (CPU)

In [ ]:
%%time
if RUN_BASELINES:
    sh(f'python run_baselines.py --data {STATIC_CSV} --out {OUT}/baselines_tinh')
    sh(f'python run_baselines.py --data {DYN_CSV} --out {OUT}/baselines_dong')
else:
    print('RUN_BASELINES = False -> bỏ qua')

## 8. Báo cáo cuối (bảng, hình, BAO_CAO.md)

In [ ]:
if RUN_REPORT:
    if os.path.exists(f'{OUT}/model/predictions_test.csv') and os.path.exists(f'{OUT}/baselines_tinh/predictions_test.csv'):
        sh(f'python final_report.py --tabpfn {OUT}/model --baselines {OUT}/baselines_tinh --data {STATIC_CSV} --oracle ./data/oracle_truth.csv --out {OUT}/bao_cao_tinh --title "Mô hình tĩnh"')
    if os.path.exists(f'{OUT}/model_dynamic/predictions_test.csv') and os.path.exists(f'{OUT}/baselines_dong/predictions_test.csv'):
        sh(f'python final_report.py --tabpfn {OUT}/model_dynamic --baselines {OUT}/baselines_dong --data {DYN_CSV} --out {OUT}/bao_cao_dong --title "Mô hình động"')
else:
    print('RUN_REPORT = False -> bỏ qua')

In [ ]:
for n in ['bao_cao_tinh', 'bao_cao_dong']:
    p = f'{OUT}/{n}/BAO_CAO.md'
    if os.path.exists(p):
        print('=' * 20, n, '=' * 20); print(open(p, encoding='utf-8').read()[:4500])

## 9. (Tuỳ chọn) Đánh giá giảng viên và Khoa chưa từng thấy
Bật `RUN_UNSEEN_EVAL`. Rất chậm vì huấn luyện lại TabPFN nhiều lần (ước lượng khoảng 1 giờ với 1 túi).

In [ ]:
%%time
if RUN_UNSEEN_EVAL:
    sh(f'python evaluate_model.py --data {STATIC_CSV} --model {OUT}/model/risk_model.joblib --out {OUT}/eval_tinh --n-bags 1')
else:
    print('RUN_UNSEEN_EVAL = False -> bỏ qua')

## 10. Bảng tóm tắt dán vào README (tự tạo từ kết quả thật)

In [ ]:
rows = []
for n, title in [('model', 'Tĩnh'), ('model_dynamic', 'Động')]:
    p = f'{OUT}/{n}/metrics.json'
    if not os.path.exists(p): continue
    m = json.load(open(p, encoding='utf-8')); t = m['test']; ci = m['test_auc_ci95']
    rows.append(f"| {title} | TabPFN {m['meta']['versions'].get('tabpfn')} ({m['meta']['settings']['n_bags']} túi, {m['device']}) | {t['roc_auc']:.4f} [{ci[0]:.3f}; {ci[1]:.3f}] | {t['pr_auc']:.4f} | {t['brier']:.4f} | {t['ece']:.4f} | {m['thresholds']} |")
md_tab = '| Mô hình | Backend | AUC test [KTC 95% theo giảng viên] | PR-AUC | Brier | ECE | Ngưỡng |\n|---|---|---|---|---|---|---|\n' + '\n'.join(rows)
open(f'{OUT}/BANG_README.md', 'w', encoding='utf-8').write(md_tab); print(md_tab)

## 11. Đóng gói để thay vào dự án (task_risk_service) và tải về

In [ ]:
import shutil
pk = f'{OUT}/goi_ket_qua'
if os.path.exists(pk): shutil.rmtree(pk)
os.makedirs(pk)
for n in ['model', 'model_dynamic']:
    if os.path.exists(f'{OUT}/{n}'):
        shutil.copytree(f'{OUT}/{n}', f'{pk}/{n}', ignore=shutil.ignore_patterns('checkpoint*'))
for n in ['bao_cao_tinh', 'bao_cao_dong', 'baselines_tinh', 'baselines_dong', 'eval_tinh']:
    if os.path.exists(f'{OUT}/{n}'): shutil.copytree(f'{OUT}/{n}', f'{pk}/{n}')
for f in ['BANG_README.md']:
    if os.path.exists(f'{OUT}/{f}'): shutil.copy(f'{OUT}/{f}', pk)
shutil.make_archive('/content/ket_qua_v2_seed44', 'zip', pk)
print('Đã tạo /content/ket_qua_v2_seed44.zip'); sh('ls -la /content/ket_qua_v2_seed44.zip')
from google.colab import files
files.download('/content/ket_qua_v2_seed44.zip')

## 12. Việc cần làm sau khi chạy
1. Kiểm tra ô "Kiểm tra nhãn mô hình": cả hai phải là `tabpfn`, `cuda`, phiên bản tabpfn rõ ràng.
2. Thay thư mục `model/` và `model_dynamic/` trong `task_risk_service` bằng bản mới trong `goi_ket_qua`; cập nhật bảng README bằng `BANG_README.md`.
3. Ghi trong báo cáo: dữ liệu mô phỏng; mô hình động học thêm từ biến tiến độ được sinh từ nhãn ẩn theo giả định; mô hình duy nhất được báo cáo là TabPFN (đã bỏ mô hình nhẹ `--fast-cpu`).
4. Ghim phiên bản `scikit-learn` trong `requirements.txt` theo `meta.versions.sklearn` của mô hình.